# lab.ipynb - Colab 비교 연구 워크벤치

mini GPT를 여러 조건으로 학습하고, loss/생성문/파인튜닝 결과를 비교하는 실험 노트북입니다.

- 기존 `src/` 파일은 수정하지 않습니다.
- 모든 학습과 테스트는 Colab에서 실행합니다.
- 기본 사용 흐름은 `SMOKE`로 점검한 뒤, 한 번에 한 변인만 바꾸며 비교하는 것입니다.


## 0. 빠른 시작

**목적**
- 처음 실행하는 사람이 전체 흐름을 빠르게 통과하도록 합니다.

**수정할 곳**
- 처음에는 `TRAINING_PRESET = "SMOKE"`를 유지합니다.
- `EXPERIMENTS`는 `learned_gelu`와 `sinusoidal_position` 비교를 그대로 둡니다.
- `PROMPTS`에 생성문을 보고 싶은 시작 문장을 추가합니다.

**출력**
- 실험 manifest 표
- train/validation loss 그래프
- token accuracy 그래프
- 학습 전/후 생성문 비교표
- CSV/PNG/Markdown 결과 파일

**해석**
- 10분 점검: `SMOKE`에서 `learned_gelu`, `sinusoidal_position`이 끝까지 도는지 확인합니다.
- 첫 비교: `learned_gelu`와 `sinusoidal_position`의 validation loss, token accuracy, 생성문, 학습 속도를 비교합니다.
- 첫 보고서: 마지막 요약 Markdown과 저장된 그래프를 보고서에 옮깁니다.


## 0.1 실험 원칙

**목적**
- 비교 실험의 기준 run을 명확히 정해 결과 해석이 흐려지지 않게 합니다.

**수정할 곳**
- 새 실험을 추가할 때는 `make_experiment("run_name", {"바꿀_값": 값})` 형식을 사용합니다.
- 결과표의 차이는 `REFERENCE_RUN_NAME`에 적은 run 이름을 기준으로 계산합니다.

**출력**
- 실험 전 manifest에서 `LEARNED_GELU_CONFIG`와 달라진 값이 표시됩니다.

**해석**
- `learned_gelu`를 먼저 완주합니다.
- 한 번에 한 변인만 바꿉니다.
- 좋은 후보만 조합 실험으로 넘깁니다.
- 최종 후보는 seed 또는 step 수를 바꿔 다시 확인합니다.


## 1. Colab 환경 설정과 저장소 준비

**목적**
- GitHub 저장소를 Colab에 clone하고, 프로젝트 모듈을 import할 수 있게 준비합니다.

**수정할 곳**
- `REPO_URL`: 저장소 주소를 미리 적어두면 입력 과정을 줄일 수 있습니다.
- `BRANCH`: 특정 브랜치를 쓸 때만 입력합니다.
- `USE_GOOGLE_DRIVE`: 긴 실험 결과를 보존하려면 `True`로 바꿉니다.

**출력**
- `Repo:` 프로젝트 경로
- `Artifacts:` 결과 저장 경로

**해석**
- `Artifacts` 경로 아래에 CSV, PNG, Markdown 결과가 저장됩니다.
- Colab 런타임이 끊길 수 있는 긴 실험은 Google Drive 저장을 권장합니다.


In [ ]:
# Colab 전용 환경 설정
import os
import subprocess
import sys
import time
from pathlib import Path

RUN_STARTED_AT = time.strftime("%Y%m%d-%H%M%S")
IN_COLAB = "google.colab" in sys.modules

# 필요하면 값을 직접 채워두세요. 비워두면 실행 시 입력창이 뜹니다.
REPO_URL = "https://github.com/Jungle-12-303/week14-team02-gpt-lab.git"  # @param {type:"string"}
BRANCH = "Helix"  # @param {type:"string"}
INSTALL_REQUIREMENTS = True  # @param {type:"boolean"}
USE_GOOGLE_DRIVE = False  # @param {type:"boolean"}
DRIVE_OUTPUT_DIR = "gpt-lab-runs"  # @param {type:"string"}


def normalize_github_url(url: str) -> str:
    url = url.strip()
    if not url:
        raise ValueError("GitHub 저장소 URL을 입력해야 합니다.")
    if url.startswith("github.com/"):
        url = "https://" + url
    if not url.startswith("https://"):
        raise ValueError("저장소 URL은 https://github.com/... 또는 github.com/... 형식이어야 합니다.")
    url = url.rstrip("/")
    if not url.endswith(".git"):
        url += ".git"
    return url


if IN_COLAB:
    from getpass import getpass

    repo_input = REPO_URL.strip() or input("GitHub 저장소 URL (예: github.com/USERNAME/gpt-lab.git): ")
    repo_url = normalize_github_url(repo_input)
    token = getpass("GitHub Personal Access Token (Private 저장소면 입력, 공개 저장소면 Enter): ").strip()
    clone_url = repo_url.replace("https://", f"https://{token}@") if token else repo_url
    repo_name = Path(repo_url[:-4]).name if repo_url.endswith(".git") else Path(repo_url).name
    repo_dir = Path("/content") / repo_name

    if not repo_dir.exists():
        clone_cmd = ["git", "clone"]
        if BRANCH.strip():
            clone_cmd += ["--branch", BRANCH.strip()]
        clone_cmd += [clone_url, str(repo_dir)]
        subprocess.run(clone_cmd, check=True)
        subprocess.run(["git", "remote", "set-url", "origin", repo_url], cwd=repo_dir, check=True)
    else:
        print(f"이미 clone된 저장소를 사용합니다: {repo_dir}")

    os.chdir(repo_dir)
else:
    repo_dir = Path(".").resolve()
    print("로컬 미리보기 모드입니다. 이 노트북의 학습/테스트 실행은 Colab에서 진행하세요.")

sys.path.insert(0, str(repo_dir / "src"))

if INSTALL_REQUIREMENTS:
    req_path = repo_dir / "requirements.txt"
    if req_path.exists():
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req_path)], check=False)

if USE_GOOGLE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ARTIFACT_DIR = Path("/content/drive/MyDrive") / DRIVE_OUTPUT_DIR / RUN_STARTED_AT
else:
    ARTIFACT_DIR = repo_dir / "checkpoints" / "lab" / RUN_STARTED_AT

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("Repo:", repo_dir)
print("Artifacts:", ARTIFACT_DIR)


## 2. 공통 import, GPU 확인, seed 고정

**목적**
- 실험에 필요한 라이브러리와 프로젝트 모듈을 불러오고 실행 환경을 확인합니다.

**수정할 곳**
- `GLOBAL_SEED`: seed 민감도를 보려면 42, 43, 44처럼 바꿔 반복 실행합니다.

**출력**
- 실행 장치, GPU 이름, PyTorch 버전

**해석**
- `Device: cuda`가 나오면 GPU 학습입니다.
- `Device: cpu`가 나오면 Colab 런타임 유형을 GPU로 바꿉니다.


In [ ]:
import copy
import csv
import json
import math
import random
import statistics
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, HTML

from bpe import BPETokenizer
from dataset import create_dataloader
from attention import MultiHeadAttention
from model import LayerNorm
from train import calc_loss_loader, generate
from finetune import ReviewSentimentDataset, GPTForSequenceClassification, train_epoch_sentiment, evaluate_sentiment


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


GLOBAL_SEED = 42
set_seed(GLOBAL_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)
print("PyTorch:", torch.__version__)


## 3. NSMC 데이터 준비와 미리보기

**목적**
- NSMC 데이터를 내려받고 LM/감성분류 파일을 준비합니다.
- 감성분류 train/validation/test의 label 분포를 먼저 확인합니다.

**수정할 곳**
- 보통 수정하지 않습니다.

**출력**
- 데이터 파일 존재 여부와 크기
- LM train corpus 미리보기
- 감성분류 split별 negative/positive 개수 preview

**해석**
- 모든 `nsmc_*` 파일이 `exists=True`여야 다음 단계로 진행할 수 있습니다.
- label 분포가 한쪽으로 치우치면 accuracy만으로 모델을 판단하지 말고 precision/recall/F1과 confusion matrix를 함께 봅니다.
- 보고서용 `data_label_distribution.csv/png`는 7번 유틸리티 셀에서 저장됩니다.


In [ ]:
import download_data

DATA_DIR = repo_dir / "data"
DATA_DIR.mkdir(exist_ok=True)

try:
    paths = download_data.main()
except Exception as e:
    print("데이터 준비 중 문제가 생겼습니다:", repr(e))
    print("이미 data/ 파일이 있다면 아래 셀부터 계속 진행할 수 있습니다.")

LM_TRAIN_PATH = DATA_DIR / "nsmc_lm_train.txt"
LM_VAL_PATH = DATA_DIR / "nsmc_lm_val.txt"
SENTIMENT_TRAIN_PATH = DATA_DIR / "nsmc_sentiment_train.jsonl"
SENTIMENT_VAL_PATH = DATA_DIR / "nsmc_sentiment_val.jsonl"
SENTIMENT_TEST_PATH = DATA_DIR / "nsmc_sentiment_test.jsonl"

for path in [LM_TRAIN_PATH, LM_VAL_PATH, SENTIMENT_TRAIN_PATH, SENTIMENT_VAL_PATH, SENTIMENT_TEST_PATH]:
    print(path.name, "exists=", path.exists(), "size=", path.stat().st_size if path.exists() else 0)

train_corpus = LM_TRAIN_PATH.read_text(encoding="utf-8") if LM_TRAIN_PATH.exists() else ""
val_corpus = LM_VAL_PATH.read_text(encoding="utf-8") if LM_VAL_PATH.exists() else ""
print("LM train chars:", len(train_corpus))
print("LM val chars:", len(val_corpus))
print("\n미리보기:")
print(train_corpus[:500])


def preview_label_counts(path: Path) -> dict[int, int]:
    """데이터 준비 단계에서 label 분포를 빠르게 확인합니다."""
    counts = {0: 0, 1: 0}
    if not path.exists():
        return counts
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            label = int(row.get("label", -1))
            if label in counts:
                counts[label] += 1
    return counts

print("\nSentiment label preview:")
for split, path in [("train", SENTIMENT_TRAIN_PATH), ("val", SENTIMENT_VAL_PATH), ("test", SENTIMENT_TEST_PATH)]:
    counts = preview_label_counts(path)
    total = counts[0] + counts[1]
    pos_share = counts[1] / max(1, total)
    print(f"{split}: negative={counts[0]:,}, positive={counts[1]:,}, positive_share={pos_share:.3f}")


## 4. 프로젝트 테스트 셀

**목적**
- 기존 과제 구현이 정상인지 Colab에서 확인합니다.

**수정할 곳**
- `RUN_PROJECT_TESTS=True`로 바꾸면 전체 테스트를 실행합니다.
- 특정 파일만 보고 싶으면 `run_pytest("tests/test_model.py")`처럼 호출합니다.

**출력**
- pytest 결과와 return code

**해석**
- return code가 0이면 기존 구현 검증 통과입니다.
- 학습 실험이 이상하면 먼저 이 셀로 코드 구현 문제인지 확인합니다.


In [ ]:
# 필요할 때 True로 바꾸고 실행하세요.
RUN_PROJECT_TESTS = False  # @param {type:"boolean"}


def run_pytest(target: str = "tests/") -> int:
    cmd = [sys.executable, "-m", "pytest", target, "-q"]
    print("실행 명령:", " ".join(cmd))
    result = subprocess.run(cmd, cwd=str(repo_dir), text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    print("return code:", result.returncode)
    return result.returncode


if RUN_PROJECT_TESTS:
    run_pytest("tests/")
else:
    print("프로젝트 테스트를 실행하려면 RUN_PROJECT_TESTS=True로 바꾸고 이 셀을 실행하세요.")


## 5. 실험 프리셋과 사용자 수정 영역

**목적**
- tokenizer, tokenized data, LM 학습 규모, fine-tuning 규모를 서로 독립적으로 정의합니다.
- 한 축을 고정한 상태에서 다른 축만 바꿔 비교할 수 있게 합니다.

**수정할 곳**
- `TOKENIZER_PRESET`, `TOKENIZER_CONFIG`: BPE vocab 크기와 BPE 학습 corpus 크기입니다.
- `TOKENIZED_DATA_PRESET`, `TOKENIZED_DATA_CONFIG`: LM train/validation 텍스트를 얼마만큼 token id로 바꿀지 정합니다.
- `TRAINING_PRESET`, `TRAINING_CONFIG`: context length, batch size, step 수 같은 LM 학습 규모입니다.
- `FINETUNE_PRESET`, `FINETUNE_CONFIG`: 감성분류 fine-tuning sample 수와 epoch 수입니다.
- `MODEL_CONFIG`, `OPTIMIZATION_CONFIG`: 모델 구조와 optimizer 설정입니다.
- `EXPERIMENTS`: 실제로 실행할 비교 실험 목록입니다.

**출력**
- 선택한 tokenizer/data/training/fine-tuning preset과 실험 목록

**해석**
- 모델 구조나 optimizer를 비교할 때는 `TOKENIZER_PRESET`과 `TOKENIZED_DATA_PRESET`을 고정합니다.
- 학습 시간을 늘리고 싶으면 `TRAINING_PRESET`만 바꿉니다. tokenizer가 같이 바뀌지 않습니다.
- tokenizer 자체를 비교할 때만 `USE_PREPARED_TOKENIZER = False`로 두고 tokenizer 실험 리스트를 실행합니다.
- `README_LIGHT`, `README_BASIC`은 배포측 권장 규모를 선택 가능한 preset으로 남겨둔 것입니다. 기본 반복 실험은 작은 preset부터 시작하세요.


In [ ]:
TOKENIZER_PRESET = "SMOKE"  # @param ["SMOKE", "LIGHT", "CORE", "LONG", "README_LIGHT", "README_BASIC", "CUSTOM"]
TOKENIZED_DATA_PRESET = "SMOKE"  # @param ["SMOKE", "LIGHT", "CORE", "LONG", "README_LIGHT", "README_BASIC", "CUSTOM"]
TRAINING_PRESET = "SMOKE"  # @param ["SMOKE", "LIGHT", "CORE", "LONG", "README_LIGHT", "README_BASIC", "CUSTOM"]
FINETUNE_PRESET = "SMOKE"  # @param ["SMOKE", "LIGHT", "CORE", "LONG", "README_LIGHT", "README_BASIC", "CUSTOM"]

TOKENIZER_PRESETS = {
    "SMOKE": {"tokenizer_vocab_size": 300, "tokenizer_train_chars": 8_000},
    "LIGHT": {"tokenizer_vocab_size": 500, "tokenizer_train_chars": 50_000},
    "CORE": {"tokenizer_vocab_size": 1_000, "tokenizer_train_chars": 200_000},
    "LONG": {"tokenizer_vocab_size": 1_500, "tokenizer_train_chars": 500_000},
    "README_LIGHT": {"tokenizer_vocab_size": 2_000, "tokenizer_train_chars": 500_000},
    "README_BASIC": {"tokenizer_vocab_size": 3_000, "tokenizer_train_chars": 1_500_000},
    "CUSTOM": {"tokenizer_vocab_size": 1_000, "tokenizer_train_chars": 200_000},
}

TOKENIZED_DATA_PRESETS = {
    "SMOKE": {"lm_train_chars": 12_000, "lm_val_chars": 4_000, "add_bos_eos_per_line": False},
    "LIGHT": {"lm_train_chars": 80_000, "lm_val_chars": 16_000, "add_bos_eos_per_line": False},
    "CORE": {"lm_train_chars": 350_000, "lm_val_chars": 60_000, "add_bos_eos_per_line": False},
    "LONG": {"lm_train_chars": 900_000, "lm_val_chars": 120_000, "add_bos_eos_per_line": False},
    "README_LIGHT": {"lm_train_chars": 500_000, "lm_val_chars": 80_000, "add_bos_eos_per_line": False},
    "README_BASIC": {"lm_train_chars": 1_500_000, "lm_val_chars": 120_000, "add_bos_eos_per_line": False},
    "CUSTOM": {"lm_train_chars": 350_000, "lm_val_chars": 60_000, "add_bos_eos_per_line": False},
}

TRAINING_PRESETS = {
    "SMOKE": {"context_length": 32, "batch_size": 8, "max_steps": 20, "eval_every": 5, "eval_batches": 2, "max_new_tokens": 30},
    "LIGHT": {"context_length": 64, "batch_size": 16, "max_steps": 150, "eval_every": 25, "eval_batches": 5, "max_new_tokens": 50},
    "CORE": {"context_length": 96, "batch_size": 24, "max_steps": 600, "eval_every": 100, "eval_batches": 10, "max_new_tokens": 80},
    "LONG": {"context_length": 128, "batch_size": 32, "max_steps": 1_500, "eval_every": 150, "eval_batches": 15, "max_new_tokens": 100},
    "README_LIGHT": {"context_length": 64, "batch_size": 16, "max_steps": 600, "eval_every": 100, "eval_batches": 10, "max_new_tokens": 80},
    "README_BASIC": {"context_length": 128, "batch_size": 16, "max_steps": 1_500, "eval_every": 150, "eval_batches": 15, "max_new_tokens": 120},
    "CUSTOM": {"context_length": 96, "batch_size": 16, "max_steps": 600, "eval_every": 100, "eval_batches": 10, "max_new_tokens": 80},
}

FINETUNE_PRESETS = {
    "SMOKE": {"finetune_train_samples": 512, "finetune_val_samples": 256, "finetune_test_samples": 256, "finetune_epochs": 1, "finetune_batch_size": 8, "finetune_max_length": 32},
    "LIGHT": {"finetune_train_samples": 2_000, "finetune_val_samples": 800, "finetune_test_samples": 800, "finetune_epochs": 2, "finetune_batch_size": 16, "finetune_max_length": 64},
    "CORE": {"finetune_train_samples": 8_000, "finetune_val_samples": 2_000, "finetune_test_samples": 2_000, "finetune_epochs": 3, "finetune_batch_size": 24, "finetune_max_length": 96},
    "LONG": {"finetune_train_samples": 20_000, "finetune_val_samples": 4_000, "finetune_test_samples": 4_000, "finetune_epochs": 3, "finetune_batch_size": 32, "finetune_max_length": 128},
    "README_LIGHT": {"finetune_train_samples": 8_000, "finetune_val_samples": 2_000, "finetune_test_samples": 2_000, "finetune_epochs": 3, "finetune_batch_size": 16, "finetune_max_length": 64},
    "README_BASIC": {"finetune_train_samples": 20_000, "finetune_val_samples": 4_000, "finetune_test_samples": 4_000, "finetune_epochs": 3, "finetune_batch_size": 16, "finetune_max_length": 128},
    "CUSTOM": {"finetune_train_samples": 8_000, "finetune_val_samples": 2_000, "finetune_test_samples": 2_000, "finetune_epochs": 3, "finetune_batch_size": 16, "finetune_max_length": 96},
}

# 필요한 경우 아래 dict만 직접 수정하세요. 다른 preset은 영향을 받지 않습니다.
TOKENIZER_CONFIG = dict(TOKENIZER_PRESETS[TOKENIZER_PRESET])
TOKENIZED_DATA_CONFIG = dict(TOKENIZED_DATA_PRESETS[TOKENIZED_DATA_PRESET])
TRAINING_CONFIG = dict(TRAINING_PRESETS[TRAINING_PRESET])
FINETUNE_CONFIG = dict(FINETUNE_PRESETS[FINETUNE_PRESET])

MODEL_CONFIG = {
    "emb_dim": 96,
    "n_heads": 4,
    "n_layers": 2,
    "drop_rate": 0.1,
    "qkv_bias": False,
    "ffn_mult": 4,
    "position_encoding": "learned",   # learned | sinusoidal | none
    "activation": "gelu",             # gelu | relu | silu | tanh
}

OPTIMIZATION_CONFIG = {
    "optimizer": "adamw",             # adamw | adam | sgd
    "learning_rate": 3e-4,
    "weight_decay": 0.01,
    "scheduler": "none",              # none | cosine_warmup
    "warmup_steps": 20,
    "grad_clip": 1.0,
    "grad_accum_steps": 1,
    "amp": True,
    "save_checkpoint": False,
}

GENERATION_PRESETS = {
    "greedy": {"temperature": 0.0, "top_k": None},
    "balanced": {"temperature": 0.8, "top_k": 40},
    "creative": {"temperature": 1.1, "top_k": 80},
}

GENERATION_CONFIG = {
    "generation_temperature": 0.8,
    "generation_top_k": 40,
}

# 이전 보고서 코드 호환용 이름입니다. 의미상으로는 TRAINING_PRESET만 나타냅니다.
RUN_PRESET = TRAINING_PRESET

LEARNED_GELU_CONFIG = {
    "seed": GLOBAL_SEED,
    **TOKENIZER_CONFIG,
    **TOKENIZED_DATA_CONFIG,
    **TRAINING_CONFIG,
    **MODEL_CONFIG,
    **OPTIMIZATION_CONFIG,
    **GENERATION_CONFIG,
}

REFERENCE_RUN_NAME = "learned_gelu"


def make_experiment(name: str, overrides: dict | None = None) -> dict:
    """LEARNED_GELU_CONFIG에서 필요한 값만 덮어써 하나의 run 설정을 만듭니다."""
    config = dict(LEARNED_GELU_CONFIG)
    config["name"] = name
    if overrides:
        config.update(overrides)
    return config


# 비교 실험은 이 리스트에 추가하세요.
EXPERIMENTS = [
    make_experiment("learned_gelu"),
    make_experiment("sinusoidal_position", {"position_encoding": "sinusoidal"}),
]

PROMPTS = [
    "이 영화는",
    "배우 연기는",
    "정말",
]

print("Tokenizer preset:", TOKENIZER_PRESET, TOKENIZER_CONFIG)
print("Tokenized data preset:", TOKENIZED_DATA_PRESET, TOKENIZED_DATA_CONFIG)
print("Training preset:", TRAINING_PRESET, TRAINING_CONFIG)
print("Fine-tuning preset:", FINETUNE_PRESET, FINETUNE_CONFIG)
for exp in EXPERIMENTS:
    print(exp["name"], "| pos=", exp["position_encoding"], "act=", exp["activation"], "lr=", exp["learning_rate"])


## 6. 실험용 모델 옵션

**목적**
- 기존 `src/`를 수정하지 않고 위치 임베딩과 활성화 함수 실험을 가능하게 합니다.

**수정할 곳**
- 보통 수정하지 않습니다.
- 새 activation이나 position encoding 방식을 추가할 때만 이 셀을 수정합니다.

**출력**
- 별도 출력은 없습니다. 이후 실험 실행 함수가 이 클래스를 사용합니다.

**해석**
- `learned`: 위치 벡터를 학습합니다.
- `sinusoidal`: 고정 sin/cos 위치 인코딩을 사용합니다.
- `none`: 위치 정보를 제거해 위치 임베딩의 필요성을 확인합니다.


In [ ]:
class LabInputEmbedding(nn.Module):
    """토큰 임베딩에 선택한 위치 정보를 더해 Transformer 입력을 만듭니다."""

    def __init__(self, vocab_size: int, emb_dim: int, context_length: int, drop_rate: float = 0.1, position_encoding: str = "learned"):
        super().__init__()
        self.emb_dim = emb_dim
        self.context_length = context_length
        self.position_encoding_type = position_encoding
        self.token_embedding = nn.Embedding(vocab_size, emb_dim)

        if position_encoding == "learned":
            # GPT 계열에서 흔히 쓰는 학습형 위치 임베딩입니다.
            self.position_embedding = nn.Embedding(context_length, emb_dim)
        elif position_encoding == "sinusoidal":
            # Transformer 원 논문의 고정 sin/cos 위치 인코딩입니다.
            position = torch.arange(context_length, dtype=torch.float).unsqueeze(1)
            div_term = torch.exp(torch.arange(0, emb_dim, 2, dtype=torch.float) * (-math.log(10000.0) / emb_dim))
            pe = torch.zeros(context_length, emb_dim)
            pe[:, 0::2] = torch.sin(position * div_term)
            pe[:, 1::2] = torch.cos(position * div_term[: pe[:, 1::2].shape[1]])
            self.register_buffer("position_encoding", pe)
        elif position_encoding == "none":
            # 위치 정보를 제거해 position encoding의 효과를 비교합니다.
            pass
        else:
            raise ValueError(f"Unknown position_encoding: {position_encoding}")

        self.dropout = nn.Dropout(drop_rate)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        _, seq_len = x.shape
        if seq_len > self.context_length:
            raise ValueError(f"seq_len={seq_len} exceeds context_length={self.context_length}")

        token_embeds = self.token_embedding(x)
        if self.position_encoding_type == "learned":
            positions = torch.arange(seq_len, device=x.device)
            embeddings = token_embeds + self.position_embedding(positions)
        elif self.position_encoding_type == "sinusoidal":
            pos = self.position_encoding[:seq_len].to(device=x.device, dtype=token_embeds.dtype)
            embeddings = token_embeds + pos
        else:
            embeddings = token_embeds
        return self.dropout(embeddings)


def make_activation(name: str) -> nn.Module:
    """문자열 설정을 실제 activation 모듈로 변환합니다."""
    name = name.lower()
    if name == "gelu":
        return nn.GELU()
    if name == "relu":
        return nn.ReLU()
    if name == "silu":
        return nn.SiLU()
    if name == "tanh":
        return nn.Tanh()
    raise ValueError(f"Unknown activation: {name}")


class LabFeedForward(nn.Module):
    """Transformer block 내부 MLP입니다. ffn_mult로 hidden 확장 배율을 바꿉니다."""

    def __init__(self, d_model: int, dropout: float = 0.1, mult: int = 4, activation: str = "gelu"):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, mult * d_model),
            make_activation(activation),
            nn.Linear(mult * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class LabTransformerBlock(nn.Module):
    """Attention과 FeedForward를 residual connection으로 묶은 GPT block입니다."""

    def __init__(self, d_model: int, n_heads: int, drop_rate: float = 0.1, qkv_bias: bool = False, ffn_mult: int = 4, activation: str = "gelu"):
        super().__init__()
        self.attention = MultiHeadAttention(d_model=d_model, n_heads=n_heads, drop_rate=drop_rate, qkv_bias=qkv_bias)
        self.ffn = LabFeedForward(d_model=d_model, dropout=drop_rate, mult=ffn_mult, activation=activation)
        self.norm1 = LayerNorm(d_model)
        self.norm2 = LayerNorm(d_model)

    def forward(self, x: torch.Tensor, causal_mask: bool = True) -> torch.Tensor:
        x = x + self.attention(self.norm1(x), causal_mask=causal_mask)
        x = x + self.ffn(self.norm2(x))
        return x


class LabGPTModel(nn.Module):
    """실험 옵션을 반영한 mini GPT 언어모델입니다."""

    def __init__(self, config: dict):
        super().__init__()
        self.config = dict(config)
        self.embedding = LabInputEmbedding(
            vocab_size=config["vocab_size"],
            emb_dim=config["emb_dim"],
            context_length=config["context_length"],
            drop_rate=config["drop_rate"],
            position_encoding=config.get("position_encoding", "learned"),
        )
        self.blocks = nn.Sequential(*[
            LabTransformerBlock(
                d_model=config["emb_dim"],
                n_heads=config["n_heads"],
                drop_rate=config["drop_rate"],
                qkv_bias=config["qkv_bias"],
                ffn_mult=config.get("ffn_mult", 4),
                activation=config.get("activation", "gelu"),
            )
            for _ in range(config["n_layers"])
        ])
        self.final_norm = LayerNorm(config["emb_dim"])
        self.lm_head = nn.Linear(config["emb_dim"], config["vocab_size"], bias=False)

    def forward(self, idx: torch.Tensor, targets: torch.Tensor | None = None):
        x = self.embedding(idx)
        x = self.blocks(x)
        x = self.final_norm(x)
        logits = self.lm_head(x)
        if targets is None:
            return logits
        loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return loss, logits


## 7. 로깅, 표, 그래프, export 유틸리티

**목적**
- 실험 결과를 화면 표, 그래프, CSV, PNG, Markdown으로 정리합니다.

**수정할 곳**
- `FIG_DPI`: 저장 그림 해상도.
- `TABLE_MAX_TEXT`: 화면 표에서 긴 텍스트를 몇 글자까지 보여줄지 정합니다.

**출력**
- 화면: 가로 스크롤이 되는 HTML 표.
- 파일: `lm_summary.csv`, `lm_history.csv`, `generation_samples.csv`, `finetune_*.csv`.
- 그림: loss/perplexity/gap/efficiency 그래프 PNG.

**해석**
- 화면 표는 빠른 확인용입니다.
- CSV는 보고서 표, 스프레드시트, 추가 분석용입니다.
- Markdown 요약은 보고서 초안으로 바로 옮기기 위한 출력입니다.


## 7.1 결과 데이터 가이드

**목적**
- 결과물을 핵심 보고서 지표와 분석/디버깅 지표로 나눠 읽습니다.
- 긴 설명은 코드 셀의 `display_result_data_guide(...)`와 `result_data_guide.md`로 출력합니다.

**출력**
- `result_data_guide.md`: 모든 결과 파일의 의미, 관찰 포인트, 해석 방법, 함께 비교할 데이터, 얻을 수 있는 통찰.
- LM 결과 섹션, fine-tuning 결과 섹션, 보고서 섹션에서 필요한 설명이 다시 표시됩니다.

**해석**
- 처음 보는 사람은 `lm_summary.csv`, 핵심 그래프, `finetune_summary.csv`부터 봅니다.
- 이상하거나 흥미로운 결과가 보이면 `lm_debug_metrics.csv`, `finetune_high_confidence_errors.csv`, `finetune_error_breakdown.csv`로 파고듭니다.


In [ ]:
RUNS = []
FINETUNE_RUNS = []
FIG_DPI = 160
TABLE_MAX_TEXT = 180

import html


LM_SUMMARY_COLUMNS = [
    "run", "config_summary", "position_encoding", "activation", "optimizer", "scheduler",
    "params", "train_tokens", "val_tokens", "tokens_seen", "approx_train_passes",
    "best_val_loss", "final_train_loss", "final_val_loss", "best_perplexity",
    "generalization_gap", "runtime", "tokens_per_sec", "reference_delta",
]
LM_HISTORY_COLUMNS = [
    "run", "step", "train_loss", "val_loss", "train_perplexity", "val_perplexity",
    "val_train_gap", "learning_rate", "elapsed_seconds", "tokens_seen_so_far",
]
LM_DEBUG_COLUMNS = [
    "run", "step", "train_token_acc", "val_token_acc", "batch_loss", "learning_rate",
    "grad_norm", "gpu_memory_mb", "tokens_seen_so_far",
]
FINETUNE_SUMMARY_COLUMNS = [
    "run", "source_lm_run", "freeze_mode", "trainable_params", "total_params",
    "train_samples", "val_samples", "test_samples", "epochs", "selected_epoch",
    "best_val_loss_epoch", "best_val_loss", "final_train_loss", "final_val_loss",
    "best_val_accuracy", "test_accuracy", "precision", "recall", "f1", "fp", "fn", "runtime",
]


RESULT_DATA_GUIDE = {
    "Data": [
        {
            "artifact": "data_label_distribution.csv",
            "role": "Dataset balance table",
            "what": "NSMC sentiment train/validation/test split별 negative/positive 개수와 비율을 정리합니다.",
            "watch": "positive_share, majority_share, imbalance_ratio를 먼저 봅니다. 한쪽 label이 많으면 accuracy가 과대평가될 수 있습니다.",
            "interpret": "분포가 거의 균형이면 accuracy 해석이 비교적 단순합니다. 분포가 치우치면 precision, recall, F1, confusion matrix를 함께 봐야 합니다.",
            "compare": "finetune_summary.csv, finetune_prf.png, confusion_{run}.png와 함께 봅니다.",
            "insight": "분류 성능 차이가 모델 개선 때문인지 label 분포와 split 차이 때문인지 구분할 수 있습니다.",
        },
        {
            "artifact": "data_label_distribution.png",
            "role": "Dataset balance chart",
            "what": "split별 negative/positive row count를 막대그래프로 보여줍니다.",
            "watch": "train, validation, test 사이에서 label 비율이 크게 달라지는지 봅니다.",
            "interpret": "validation과 test 분포가 다르면 validation 기준으로 고른 모델의 test 성능이 예상과 다를 수 있습니다.",
            "compare": "data_label_distribution.csv의 정확한 비율과 함께 봅니다.",
            "insight": "fine-tuning 결과 해석 전에 평가 데이터 자체가 공정한지 빠르게 점검할 수 있습니다.",
        },
    ],
    "LM Pretraining": [
        {
            "artifact": "lm_config_manifest.csv",
            "role": "Experiment setup table",
            "what": "실행한 LM 실험의 설정을 run별로 정리합니다. 어떤 변인이 바뀌었는지, 모델 크기와 학습 설정이 어떻게 다른지 확인하는 출발점입니다.",
            "watch": "run 이름, config_summary, parameter count, batch size, learning rate, tokenizer 설정을 먼저 봅니다.",
            "interpret": "결과 차이를 해석하기 전에 실험 설정이 의도대로 한 변인만 바뀌었는지 확인합니다. 설정이 여러 개 동시에 바뀌면 원인 해석이 어려워집니다.",
            "compare": "lm_summary.csv와 나란히 보며 어떤 설정 변화가 validation loss, perplexity, runtime 변화로 이어졌는지 연결합니다.",
            "insight": "실험 설계가 제대로 통제되었는지 확인하고, 다음 실험에서 고정할 값과 바꿀 값을 정할 수 있습니다.",
        },
        {
            "artifact": "lm_summary.csv",
            "role": "Core LM comparison table",
            "what": "LM 사전학습 run을 한 줄씩 비교하는 핵심 요약표입니다. 구조, 학습량, 품질, 비용을 한 화면에서 봅니다.",
            "watch": "best_val_loss, best_perplexity, generalization_gap, tokens_per_sec, reference_delta를 먼저 봅니다.",
            "interpret": "validation loss와 perplexity가 낮으면 다음 토큰 예측이 더 안정적입니다. gap이 크면 train 데이터에만 맞는 방향일 수 있습니다.",
            "compare": "lm_config_manifest.csv, lm_history.csv, lm_quality_vs_params.png, lm_quality_vs_runtime.png와 함께 봅니다.",
            "insight": "좋은 run은 성능만 좋은 것이 아니라 비슷한 비용에서 더 낮은 validation loss를 보입니다.",
        },
        {
            "artifact": "lm_reference_comparison.csv",
            "role": "Reference comparison table",
            "what": "REFERENCE_RUN_NAME으로 지정한 run과 각 run의 best validation loss 차이를 따로 정리합니다.",
            "watch": "reference_delta가 음수인지, 개선 폭이 비용 증가에 비해 충분한지 봅니다.",
            "interpret": "작은 음수라도 runtime이나 params가 크게 늘었다면 실용적 개선이 아닐 수 있습니다. 양수면 reference보다 validation loss가 나빠진 것입니다.",
            "compare": "lm_quality_vs_runtime.png, lm_quality_vs_params.png, lm_summary.csv의 params/runtime과 같이 봅니다.",
            "insight": "단순히 가장 낮은 loss가 아니라 비용 대비 개선이 큰 설정을 고를 수 있습니다.",
        },
        {
            "artifact": "lm_history.csv",
            "role": "Core LM learning trace",
            "what": "평가가 수행된 optimizer step마다 train/validation loss, perplexity, val-train gap, learning rate를 기록합니다.",
            "watch": "loss가 꾸준히 내려가는지, validation loss가 어느 시점부터 멈추거나 올라가는지 봅니다.",
            "interpret": "train loss만 계속 내려가고 validation loss가 정체되면 overfitting이나 데이터 부족을 의심합니다. 둘 다 높으면 undertraining이나 모델 용량 부족일 수 있습니다.",
            "compare": "lm_loss.png, lm_perplexity.png, lm_val_train_gap.png의 원본 데이터입니다.",
            "insight": "좋은 설정은 같은 step 안에서 더 빠르게 validation loss를 낮추고, gap이 과하게 벌어지지 않습니다.",
        },
        {
            "artifact": "lm_loss.png",
            "role": "Learning curve",
            "what": "x축은 실제 평가가 수행된 optimizer step, y축은 cross entropy loss입니다. train과 validation loss를 run별로 비교합니다.",
            "watch": "validation loss가 내려가는 속도, 최저점 이후 반등, train과 validation 곡선 간격을 봅니다.",
            "interpret": "validation loss가 안정적으로 내려가면 학습이 진행 중입니다. train만 내려가고 validation이 올라가면 과적합 가능성이 큽니다.",
            "compare": "lm_val_train_gap.png와 함께 gap의 방향을 확인하고, lm_summary.csv의 best_val_loss와 연결합니다.",
            "insight": "학습 step을 늘릴지, learning rate를 줄일지, 모델 크기를 조정할지 판단할 수 있습니다.",
        },
        {
            "artifact": "lm_perplexity.png",
            "role": "Prediction uncertainty curve",
            "what": "x축은 optimizer step, y축은 validation perplexity입니다. perplexity는 validation loss를 지수 변환한 값입니다.",
            "watch": "낮을수록 모델이 다음 토큰 후보를 덜 헷갈립니다. run 간 상대 순위와 하락 속도를 봅니다.",
            "interpret": "loss보다 직관적인 품질 지표지만 tokenizer가 달라지면 직접 비교가 조심스럽습니다.",
            "compare": "lm_loss.png와 같은 방향인지 확인하고, generation_samples.csv의 정성 결과와 함께 봅니다.",
            "insight": "비슷한 loss 차이가 실제 예측 불확실성에서 얼마나 커 보이는지 설명할 수 있습니다.",
        },
        {
            "artifact": "lm_val_train_gap.png",
            "role": "Generalization gap curve",
            "what": "x축은 optimizer step, y축은 validation loss minus train loss입니다. 값이 클수록 train보다 validation이 어렵다는 뜻입니다.",
            "watch": "gap이 계속 커지는지, 특정 run만 gap이 급격히 벌어지는지 봅니다.",
            "interpret": "gap이 크면 과적합, 데이터 분포 차이, validation set 난이도 차이를 의심합니다. gap이 작아도 loss가 높으면 아직 충분히 배우지 못한 상태일 수 있습니다.",
            "compare": "lm_loss.png와 함께 봐야 합니다. gap만 작다고 좋은 모델은 아닙니다.",
            "insight": "regularization, 모델 크기, 학습 step, 데이터 분할을 조정할 근거를 얻습니다.",
        },
        {
            "artifact": "lm_quality_vs_params.png",
            "role": "Quality versus model size",
            "what": "x축은 parameter count, y축은 best validation loss입니다. 모델 크기 증가가 품질 개선으로 이어졌는지 봅니다.",
            "watch": "큰 모델이 실제로 더 낮은 loss를 얻었는지, 작은 모델이 비용 대비 좋은 지점인지 봅니다.",
            "interpret": "큰 모델이 더 나쁘면 step 부족, learning rate 부적합, batch/tokenizer 차이, seed 변동을 점검합니다.",
            "compare": "lm_quality_vs_runtime.png, lm_quality_vs_throughput.png와 함께 비용 효율을 봅니다.",
            "insight": "T4 환경에서 감당 가능한 모델 크기와 성능 균형점을 찾을 수 있습니다.",
        },
        {
            "artifact": "lm_quality_vs_runtime.png",
            "role": "Quality versus wall-clock cost",
            "what": "x축은 runtime seconds, y축은 best validation loss입니다. 학습 시간이 품질로 얼마나 전환되었는지 봅니다.",
            "watch": "오래 돌렸는데 loss 개선이 작거나 없는 run을 찾습니다.",
            "interpret": "runtime이 길수록 좋은 것은 아닙니다. 같은 시간에서 loss가 낮은 run이 효율적입니다.",
            "compare": "lm_quality_vs_params.png와 함께 보면 모델 크기 때문인지 학습 효율 때문인지 분리해 볼 수 있습니다.",
            "insight": "긴 실험으로 확장할 후보와 중단할 후보를 정할 수 있습니다.",
        },
        {
            "artifact": "lm_quality_vs_throughput.png",
            "role": "Quality versus token throughput",
            "what": "x축은 tokens/sec, y축은 best validation loss입니다. 처리량과 품질의 균형을 봅니다.",
            "watch": "처리량은 높은데 품질이 낮은 run, 품질은 좋지만 너무 느린 run을 구분합니다.",
            "interpret": "throughput은 batch size, context length, model size, GPU 상태의 영향을 받습니다. 단독 성능 지표가 아니라 비용 지표입니다.",
            "compare": "lm_summary.csv의 tokens_seen, runtime, params와 함께 봅니다.",
            "insight": "Colab T4에서 반복 실험을 많이 돌릴 수 있는 현실적인 설정을 고를 수 있습니다.",
        },
        {
            "artifact": "generation_samples.csv",
            "role": "Raw qualitative samples",
            "what": "같은 prompt를 before/after, decoding preset별로 넣어 생성문을 저장합니다. temperature와 top_k도 함께 기록합니다.",
            "watch": "after가 before보다 자연스러운지, 반복 출력이나 깨진 문자, prompt 무시가 줄었는지 봅니다.",
            "interpret": "생성문은 정성 평가입니다. 좋아 보여도 validation loss가 나쁘면 우연한 샘플일 수 있습니다.",
            "compare": "lm_summary.csv의 best_val_loss, lm_perplexity.png, lm_generation_review.md와 함께 봅니다.",
            "insight": "수치가 비슷한 run끼리 실제 사용감 차이를 발견할 수 있습니다.",
        },
        {
            "artifact": "lm_generation_review.md",
            "role": "Readable generation review",
            "what": "generation_samples.csv를 prompt별 Markdown 표로 정리해 보고서에 붙이기 쉽게 만든 파일입니다.",
            "watch": "동일 prompt에서 run, phase, decoding별 차이를 행 단위로 읽습니다.",
            "interpret": "balanced와 creative decoding은 모델 품질뿐 아니라 샘플링 설정의 영향을 받습니다. 수치 지표와 분리해서 읽습니다.",
            "compare": "generation_samples.csv 원본과 lm_summary.csv의 validation 지표를 함께 봅니다.",
            "insight": "보고서에서 정량 지표만으로 설명하기 어려운 품질 차이를 예시로 보여줄 수 있습니다.",
        },
        {
            "artifact": "lm_debug_metrics.csv",
            "role": "LM debug table",
            "what": "token accuracy, batch loss, gradient norm, GPU memory, tokens_seen_so_far 같은 sanity/debug 지표입니다.",
            "watch": "token accuracy가 전혀 오르지 않는지, grad_norm이 튀는지, memory가 설정 변경과 함께 급증하는지 봅니다.",
            "interpret": "LM의 핵심 평가지표는 loss/perplexity입니다. token accuracy는 학습이 망가졌는지 확인하는 보조 지표로 봅니다.",
            "compare": "lm_history.csv의 loss 곡선과 같이 봐야 합니다.",
            "insight": "loss가 이상하게 움직일 때 optimizer, lr, batch, model size 문제를 빠르게 좁힐 수 있습니다.",
        },
        {
            "artifact": "lm_debug_token_accuracy.png",
            "role": "LM debug curve",
            "what": "x축은 optimizer step, y축은 next-token top-1 accuracy입니다. 다음 token id를 정확히 맞춘 비율입니다.",
            "watch": "accuracy가 0 근처에서 움직이지 않거나 갑자기 무너지는지 봅니다.",
            "interpret": "tokenizer가 바뀌면 token 단위가 달라지므로 run 간 직접 비교가 제한됩니다. 품질 순위는 loss/perplexity로 판단합니다.",
            "compare": "lm_loss.png와 함께 봅니다. loss가 내려가는데 token accuracy가 조금만 오르는 것은 작은 LM에서 자연스러울 수 있습니다.",
            "insight": "학습 루프가 실제로 신호를 받고 있는지 빠르게 확인할 수 있습니다.",
        },
        {
            "artifact": "lm_report_summary.md",
            "role": "LM-only report block",
            "what": "LM pretraining 결과만 따로 떼어 보고서에 붙일 수 있도록 요약합니다.",
            "watch": "핵심 표, reference comparison, 생성문 리뷰, 다음 LM 실험 후보를 순서대로 읽습니다.",
            "interpret": "LM 파트만 공유할 때는 이 파일을 사용하면 fine-tuning 결과와 섞이지 않습니다.",
            "compare": "lab_report_summary.md의 전체 결론과 비교해 LM 결과가 fine-tuning으로 이어졌는지 확인합니다.",
            "insight": "사전학습 단계에서 어떤 설정을 다음 backbone 후보로 삼을지 설명할 수 있습니다.",
        },
    ],
    "Fine-tuning": [
        {
            "artifact": "finetune_config_manifest.csv",
            "role": "Fine-tuning setup table",
            "what": "fine-tuning run별 source LM, freeze mode, trainable parameter 수, learning rate, epoch 설정을 정리합니다.",
            "watch": "freeze_mode와 trainable_params가 의도한 실험 범위와 맞는지 확인합니다.",
            "interpret": "성능 차이를 보기 전에 실제로 classifier만 학습했는지, full fine-tuning인지 검증합니다.",
            "compare": "finetune_summary.csv와 함께 보며 튜닝 범위가 accuracy/F1과 비용에 미친 영향을 봅니다.",
            "insight": "파인튜닝 실험에서 바뀐 변인이 무엇인지 명확하게 설명할 수 있습니다.",
        },
        {
            "artifact": "finetune_summary.csv",
            "role": "Core task table",
            "what": "감성분류 fine-tuning run의 task metric과 비용을 요약합니다.",
            "watch": "best_val_accuracy, test_accuracy, precision, recall, F1, FP/FN을 먼저 봅니다.",
            "interpret": "accuracy가 높아도 precision/recall 한쪽이 무너지면 특정 class에 약한 모델일 수 있습니다.",
            "compare": "finetune_scores.png, finetune_prf.png, confusion matrix와 함께 봅니다.",
            "insight": "freeze 범위나 learning rate 변화가 실제 task 성능을 개선했는지 판단할 수 있습니다.",
        },
        {
            "artifact": "finetune_history.csv",
            "role": "Task learning trace",
            "what": "epoch별 train/validation loss, accuracy, val-train loss gap, learning rate, elapsed_seconds를 기록합니다.",
            "watch": "val accuracy가 언제 최고점을 찍는지, val-train loss gap이 커지는지 봅니다.",
            "interpret": "train accuracy만 오르고 val accuracy가 정체되면 overfitting 가능성이 큽니다.",
            "compare": "finetune_loss.png, finetune_accuracy_curve.png의 원본 데이터입니다.",
            "insight": "epoch 수, fine-tuning lr, freeze mode를 줄이거나 늘릴 근거를 얻습니다.",
        },
        {
            "artifact": "finetune_predictions.csv",
            "role": "Row-level prediction audit",
            "what": "test 문장별 정답, 예측, confidence, prob_negative, prob_positive, 오답 유형을 저장합니다.",
            "watch": "confidence가 높은 오답과 한쪽 class로 치우친 예측을 봅니다.",
            "interpret": "높은 confidence로 틀린 예시는 모델이 잘못 배운 패턴을 보여줍니다.",
            "compare": "finetune_high_confidence_errors.csv, finetune_error_breakdown.csv와 함께 봅니다.",
            "insight": "데이터 추가, tokenizer 점검, freeze 범위 조정의 방향을 잡을 수 있습니다.",
        },
        {
            "artifact": "finetune_high_confidence_errors.csv",
            "role": "Hard error examples",
            "what": "confidence가 높은 오답만 모아 정리합니다. 모델이 확신했지만 틀린 사례입니다.",
            "watch": "반어법, 짧은 문장, 욕설, 중립에 가까운 표현, label noise 같은 반복 패턴을 찾습니다.",
            "interpret": "모델 구조보다 데이터 품질이나 task 정의가 문제인 경우를 드러낼 수 있습니다.",
            "compare": "finetune_predictions.csv 원본과 confusion matrix를 함께 봅니다.",
            "insight": "성능 개선을 위해 모델을 키울지, 데이터를 정제할지, 평가 기준을 바꿀지 판단할 수 있습니다.",
        },
        {
            "artifact": "finetune_error_breakdown.csv",
            "role": "Error direction table",
            "what": "FP와 FN이 각각 몇 개이고 전체 오류에서 차지하는 비율이 얼마인지 정리합니다.",
            "watch": "negative를 positive로 틀리는 FP가 많은지, positive를 negative로 놓치는 FN이 많은지 봅니다.",
            "interpret": "FP가 많으면 긍정 과잉, FN이 많으면 긍정 회수 부족 경향입니다.",
            "compare": "precision, recall, confusion_{run}.png와 함께 봅니다.",
            "insight": "모델의 오류 성향을 설명하고 다음 데이터 보강 방향을 제안할 수 있습니다.",
        },
        {
            "artifact": "finetune_loss.png",
            "role": "Fine-tuning loss curve",
            "what": "x축은 epoch, y축은 train/validation loss입니다. fine-tuning이 안정적으로 진행되는지 봅니다.",
            "watch": "validation loss 최저점, train loss와 validation loss의 벌어짐, epoch 후반 반등을 봅니다.",
            "interpret": "validation loss가 반등하면 더 오래 학습하는 것이 오히려 나쁠 수 있습니다.",
            "compare": "finetune_accuracy_curve.png와 함께 봅니다. loss는 나빠져도 accuracy가 유지되는 경우가 있습니다.",
            "insight": "early stopping, epoch 수, learning rate 조정 근거를 얻습니다.",
        },
        {
            "artifact": "finetune_accuracy_curve.png",
            "role": "Fine-tuning accuracy curve",
            "what": "x축은 epoch, y축은 train/validation accuracy입니다. epoch 진행에 따른 task score 변화를 봅니다.",
            "watch": "validation accuracy가 빨리 포화되는지, train accuracy만 계속 올라가는지 봅니다.",
            "interpret": "train과 validation accuracy 차이가 커지면 과적합 가능성이 있습니다.",
            "compare": "finetune_loss.png, finetune_history.csv와 같이 봅니다.",
            "insight": "classifier_only와 full_finetune 중 어떤 튜닝 범위가 더 안정적인지 판단할 수 있습니다.",
        },
        {
            "artifact": "finetune_scores.png",
            "role": "Task score comparison",
            "what": "run별 best validation accuracy, test accuracy, F1을 한 그래프에서 비교합니다.",
            "watch": "validation 성능과 test 성능이 같은 방향인지, F1이 accuracy보다 낮게 떨어지는지 봅니다.",
            "interpret": "validation만 좋고 test가 낮으면 과적합 또는 validation 분할 운을 의심합니다.",
            "compare": "finetune_summary.csv와 confusion matrix를 함께 봅니다.",
            "insight": "최종 후보를 고를 때 단일 metric에 끌려가지 않도록 도와줍니다.",
        },
        {
            "artifact": "finetune_prf.png",
            "role": "Precision/recall/F1 comparison",
            "what": "run별 precision, recall, F1을 비교합니다. class error 균형을 확인하는 그래프입니다.",
            "watch": "precision과 recall 중 한쪽만 높고 다른 쪽이 낮은 run을 찾습니다.",
            "interpret": "precision이 낮으면 positive 예측 중 오답이 많고, recall이 낮으면 실제 positive를 많이 놓칩니다.",
            "compare": "finetune_error_breakdown.csv, confusion_{run}.png와 함께 봅니다.",
            "insight": "accuracy가 비슷한 run 중 실제 사용 목적에 맞는 모델을 고를 수 있습니다.",
        },
        {
            "artifact": "finetune_quality_vs_trainable_params.png",
            "role": "Task quality versus trainable size",
            "what": "x축은 trainable parameter count, y축은 test accuracy입니다. 더 많이 학습할수록 task 성능이 좋아지는지 봅니다.",
            "watch": "full fine-tuning이 classifier_only보다 비용 대비 충분히 좋아졌는지 봅니다.",
            "interpret": "trainable parameter가 많아도 성능이 비슷하면 더 단순한 freeze mode가 실용적일 수 있습니다.",
            "compare": "finetune_quality_vs_runtime.png, finetune_summary.csv의 runtime과 함께 봅니다.",
            "insight": "제한된 Colab 시간에서 어떤 fine-tuning 범위가 효율적인지 고를 수 있습니다.",
        },
        {
            "artifact": "finetune_quality_vs_runtime.png",
            "role": "Task quality versus runtime",
            "what": "x축은 runtime seconds, y축은 test accuracy입니다. 학습 시간이 task 성능으로 얼마나 전환됐는지 봅니다.",
            "watch": "오래 걸렸는데 test accuracy 개선이 작은 run을 찾습니다.",
            "interpret": "비슷한 성능이면 짧은 runtime의 설정이 반복 실험에 유리합니다.",
            "compare": "finetune_quality_vs_trainable_params.png와 함께 봅니다.",
            "insight": "추가 실험을 빠르게 반복할 후보와 긴 검증을 할 후보를 분리할 수 있습니다.",
        },
        {
            "artifact": "confusion_{run}.png",
            "role": "Raw confusion matrix",
            "what": "x축은 predicted label, y축은 true label입니다. 각 칸은 test set에서의 실제 개수입니다.",
            "watch": "대각선은 정답 수, 오른쪽 위는 FP, 왼쪽 아래는 FN입니다.",
            "interpret": "raw count는 실제 오류 개수를 보여주므로 오류 표본을 얼마나 확보했는지 판단하기 좋습니다.",
            "compare": "finetune_error_breakdown.csv, finetune_prf.png와 함께 봅니다.",
            "insight": "모델이 어느 방향으로 틀리는지 한눈에 설명할 수 있습니다.",
        },
        {
            "artifact": "confusion_{run}_normalized.png",
            "role": "Normalized confusion matrix",
            "what": "각 true label 행을 1로 정규화한 confusion matrix입니다. 클래스별 오류 비율을 보여줍니다.",
            "watch": "label 수가 불균형할 때 특정 class에서 오류율이 높은지 봅니다.",
            "interpret": "raw count가 작아도 normalized 비율이 높으면 그 class에서 취약할 수 있습니다.",
            "compare": "confusion_{run}.png와 함께 count와 rate를 같이 봅니다.",
            "insight": "데이터 분포 차이에 가려진 클래스별 약점을 발견할 수 있습니다.",
        },
        {
            "artifact": "finetune_report_summary.md",
            "role": "Fine-tuning-only report block",
            "what": "fine-tuning 결과와 오류 분석만 따로 보고서에 붙일 수 있도록 요약합니다.",
            "watch": "task metric, FP/FN breakdown, high-confidence errors, 다음 fine-tuning 후보를 순서대로 봅니다.",
            "interpret": "분류 task 관점의 결론만 공유할 때 사용합니다.",
            "compare": "lm_report_summary.md와 함께 보면 좋은 LM backbone이 실제 task에서도 좋은지 확인할 수 있습니다.",
            "insight": "사전학습 품질과 downstream task 성능이 같은 방향인지 검증할 수 있습니다.",
        },
    ],
    "Report/Guide": [
        {
            "artifact": "lab_report_summary.md",
            "role": "Full report summary",
            "what": "LM, generation, fine-tuning, error analysis, next experiments를 한 문서로 정리합니다.",
            "watch": "best run만 보지 말고 reference_delta, runtime, generation sample, error 방향을 함께 봅니다.",
            "interpret": "보고서의 결론은 metric, 생성문, 오류 분석이 같은 방향을 가리킬 때 가장 설득력 있습니다.",
            "compare": "lm_report_summary.md, finetune_report_summary.md를 필요한 파트별로 나눠 봅니다.",
            "insight": "다음 실험 후보를 명확하게 정리하고, 왜 그 실험이 필요한지 설명할 수 있습니다.",
        },
        {
            "artifact": "lm_report_summary.md",
            "role": "LM report extract",
            "what": "전체 보고서에서 LM pretraining 파트만 분리한 요약입니다.",
            "watch": "설정표, metric 표, 생성문 비교, LM 다음 실험 후보를 봅니다.",
            "interpret": "모델 구조나 학습 방식 비교만 논의할 때 사용합니다.",
            "compare": "lab_report_summary.md에서 fine-tuning 결과와 연결합니다.",
            "insight": "사전학습 단계의 좋은 후보가 downstream task에서도 좋은지 따로 검증할 수 있습니다.",
        },
        {
            "artifact": "finetune_report_summary.md",
            "role": "Fine-tuning report extract",
            "what": "전체 보고서에서 fine-tuning과 오류 분석 파트만 분리한 요약입니다.",
            "watch": "accuracy/F1, confusion matrix 방향, high-confidence errors, fine-tuning 다음 후보를 봅니다.",
            "interpret": "감성분류 task 관점에서 모델 선택 근거를 설명할 때 사용합니다.",
            "compare": "lm_report_summary.md의 LM 후보 순위와 비교합니다.",
            "insight": "사전학습 loss가 낮은 모델이 실제 분류에서도 좋은지 확인할 수 있습니다.",
        },
        {
            "artifact": "result_data_guide.md",
            "role": "Reader guide",
            "what": "각 결과 파일이 무엇을 보여주는지, 무엇을 포착해야 하는지, 어떻게 비교할지 설명합니다.",
            "watch": "처음 보는 사람에게는 이 문서를 먼저 보여주면 결과 해석 속도가 빨라집니다.",
            "interpret": "결과 파일은 단독으로보다 서로 연결해 읽을 때 의미가 커집니다.",
            "compare": "핵심 표, 학습 곡선, 생성문, 오류 분석을 순서대로 연결합니다.",
            "insight": "여러 실험 결과를 단순 순위가 아니라 연구 질문으로 바꿀 수 있습니다.",
        },
    ],
}

def count_parameters(model: nn.Module, trainable_only: bool = False) -> int:
    """모델 파라미터 수를 계산합니다. trainable_only=True면 학습되는 파라미터만 셉니다."""
    params = model.parameters()
    if trainable_only:
        return sum(p.numel() for p in params if p.requires_grad)
    return sum(p.numel() for p in params)


def estimate_model_parameters(config: dict) -> int:
    """실험 전 manifest에서 모델 크기를 미리 보여주기 위한 가벼운 추정 함수입니다."""
    cfg = dict(config)
    prepared_tokenizer = globals().get("PREPARED_TOKENIZER")
    if globals().get("USE_PREPARED_TOKENIZER", False) and prepared_tokenizer is not None:
        cfg["vocab_size"] = len(prepared_tokenizer.id_to_token)
    else:
        cfg["vocab_size"] = config.get("actual_vocab_size", config.get("tokenizer_vocab_size", 300))
    model_cfg = model_config_from_experiment(cfg) if "model_config_from_experiment" in globals() else {
        "vocab_size": cfg["vocab_size"],
        "context_length": cfg["context_length"],
        "emb_dim": cfg["emb_dim"],
        "n_heads": cfg["n_heads"],
        "n_layers": cfg["n_layers"],
        "drop_rate": cfg["drop_rate"],
        "qkv_bias": cfg["qkv_bias"],
        "ffn_mult": cfg.get("ffn_mult", 4),
        "position_encoding": cfg.get("position_encoding", "learned"),
        "activation": cfg.get("activation", "gelu"),
    }
    was_device = globals().get("device", torch.device("cpu"))
    model = LabGPTModel(model_cfg).to("cpu")
    n_params = count_parameters(model)
    del model
    if was_device.type == "cuda":
        torch.cuda.empty_cache()
    return n_params


def perplexity(loss: float) -> float:
    """cross entropy loss를 perplexity로 변환합니다."""
    if loss is None or math.isnan(loss):
        return float("nan")
    return float(math.exp(min(20.0, loss)))


def format_seconds(seconds: float) -> str:
    seconds = int(seconds)
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    if h:
        return f"{h}h {m}m {s}s"
    if m:
        return f"{m}m {s}s"
    return f"{s}s"


def format_float(value, digits: int = 3) -> str:
    if value is None:
        return ""
    try:
        if math.isnan(float(value)):
            return ""
        return f"{float(value):.{digits}f}"
    except Exception:
        return str(value)


def md_escape(value) -> str:
    """Markdown 파일로 내보낼 때 셀 안의 특수문자와 줄바꿈을 안전하게 바꿉니다."""
    text = "" if value is None else str(value)
    return text.replace("|", "\\|").replace(chr(10), "<br>")


def markdown_table(rows: list[dict], columns: list[tuple[str, str]]) -> list[str]:
    """보고서용 Markdown 표 문자열 리스트를 만듭니다."""
    if not rows:
        return ["_No rows._"]
    header = "| " + " | ".join(label for _, label in columns) + " |"
    sep = "| " + " | ".join("---" for _ in columns) + " |"
    body = ["| " + " | ".join(md_escape(row.get(key, "")) for key, _ in columns) + " |" for row in rows]
    return [header, sep] + body


def html_cell(value, max_text: int = TABLE_MAX_TEXT) -> str:
    """화면 표시용 HTML 셀을 만듭니다. 긴 텍스트는 줄여서 표가 무너지지 않게 합니다."""
    text = "" if value is None else str(value)
    if len(text) > max_text:
        text = text[:max_text] + "..."
    return html.escape(text).replace(chr(10), "<br>")


def display_table(rows: list[dict], columns: list[tuple[str, str]], title: str | None = None) -> None:
    """Colab에서 읽기 쉬운 HTML 표로 표시합니다. CSV export는 원본 값을 그대로 저장합니다."""
    if not rows:
        title_html = f"<h3>{html.escape(title)}</h3>" if title else ""
        display(HTML(title_html + "<p><em>No rows.</em></p>"))
        return

    style = """
    <style>
    .lab-table-wrap { overflow-x: auto; max-width: 100%; margin: 0.5rem 0 1rem 0; }
    table.lab-table { border-collapse: collapse; font-size: 13px; min-width: 760px; }
    table.lab-table th, table.lab-table td { border: 1px solid #ddd; padding: 6px 8px; vertical-align: top; }
    table.lab-table th { background: #f6f8fa; font-weight: 600; white-space: nowrap; }
    table.lab-table td { max-width: 360px; white-space: normal; word-break: break-word; }
    </style>
    """
    title_html = f"<h3>{html.escape(title)}</h3>" if title else ""
    header = "".join(f"<th>{html.escape(label)}</th>" for _, label in columns)
    body_rows = []
    for row in rows:
        cells = "".join(f"<td>{html_cell(row.get(key, ''))}</td>" for key, _ in columns)
        body_rows.append(f"<tr>{cells}</tr>")
    table_html = f"<div class='lab-table-wrap'><table class='lab-table'><thead><tr>{header}</tr></thead><tbody>{''.join(body_rows)}</tbody></table></div>"
    display(HTML(style + title_html + table_html))


def write_csv(path: Path, rows: list[dict], columns: list[str] | None = None) -> None:
    """보고서와 스프레드시트에서 쓰기 쉬운 CSV 파일을 저장합니다."""
    if columns is None:
        keys = []
        for row in rows:
            for key in row.keys():
                if key not in keys:
                    keys.append(key)
        columns = keys
    with path.open("w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=columns)
        writer.writeheader()
        for row in rows:
            writer.writerow({key: row.get(key, "") for key in columns})


def write_markdown(path: Path, text: str) -> Path:
    path.write_text(text, encoding="utf-8")
    print("saved:", path)
    return path


def save_jsonl(path: Path, rows: list[dict]) -> None:
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + chr(10))


def save_current_figure(filename: str) -> Path:
    """현재 matplotlib figure를 artifact 폴더에 저장합니다."""
    path = ARTIFACT_DIR / filename
    plt.savefig(path, dpi=FIG_DPI, bbox_inches="tight")
    print("saved figure:", path)
    return path


def sentiment_rows_for_distribution(path: Path) -> list[dict]:
    """Read only the fields needed for label distribution checks."""
    rows = []
    if not path.exists():
        return rows
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows


def data_label_distribution_rows() -> list[dict]:
    """Build split-level label distribution rows for reports and plots."""
    required = ["SENTIMENT_TRAIN_PATH", "SENTIMENT_VAL_PATH", "SENTIMENT_TEST_PATH"]
    if not all(name in globals() for name in required):
        return []
    split_paths = {
        "train": SENTIMENT_TRAIN_PATH,
        "validation": SENTIMENT_VAL_PATH,
        "test": SENTIMENT_TEST_PATH,
    }
    rows = []
    for split, path in split_paths.items():
        raw_rows = sentiment_rows_for_distribution(path)
        total = len(raw_rows)
        counts = {
            0: sum(1 for row in raw_rows if int(row.get("label", -1)) == 0),
            1: sum(1 for row in raw_rows if int(row.get("label", -1)) == 1),
        }
        nonzero_counts = [count for count in counts.values() if count > 0]
        majority_share = max(counts.values()) / max(1, total)
        imbalance_ratio = max(nonzero_counts) / min(nonzero_counts) if len(nonzero_counts) == 2 else None
        for label, label_name in [(0, "negative"), (1, "positive")]:
            rows.append({
                "split": split,
                "label": label,
                "label_name": label_name,
                "count": counts[label],
                "total": total,
                "share": counts[label] / max(1, total),
                "positive_share": counts[1] / max(1, total),
                "majority_share": majority_share,
                "imbalance_ratio": imbalance_ratio,
                "source_path": str(path),
            })
    return rows


def plot_data_label_distribution(rows: list[dict], save: bool = True) -> None:
    """Plot negative/positive counts by split."""
    if not rows:
        print("No label distribution rows to plot.")
        return
    splits = []
    for row in rows:
        if row["split"] not in splits:
            splits.append(row["split"])
    negative = [next((r["count"] for r in rows if r["split"] == split and r["label"] == 0), 0) for split in splits]
    positive = [next((r["count"] for r in rows if r["split"] == split and r["label"] == 1), 0) for split in splits]
    x = np.arange(len(splits))
    width = 0.35
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(x - width / 2, negative, width, label="negative")
    ax.bar(x + width / 2, positive, width, label="positive")
    ax.set_xticks(x)
    ax.set_xticklabels(splits)
    ax.set_xlabel("split")
    ax.set_ylabel("row count")
    ax.set_title("Sentiment label distribution")
    ax.legend()
    ax.grid(axis="y", alpha=0.25)
    if save:
        save_current_figure("data_label_distribution.png")
    plt.show()


def export_data_label_distribution(show_table: bool = True, save_plot: bool = True) -> list[dict]:
    """Save data balance artifacts and optionally display the summary table."""
    rows = data_label_distribution_rows()
    if not rows:
        print("No sentiment label distribution rows available.")
        return []
    write_csv(ARTIFACT_DIR / "data_label_distribution.csv", rows)
    if show_table:
        display_table(rows, [("split", "split"), ("label_name", "label"), ("count", "count"), ("share", "share"), ("positive_share", "positive share"), ("majority_share", "majority share"), ("imbalance_ratio", "imbalance ratio")], title="Data label distribution")
    if save_plot:
        plot_data_label_distribution(rows, save=True)
    return rows


def result_data_guide_markdown(sections: str | list[str] | None = None) -> str:
    if sections is None:
        selected = list(RESULT_DATA_GUIDE.keys())
    elif isinstance(sections, str):
        selected = [sections]
    else:
        selected = sections

    lines = ["# Result Data Guide", ""]
    for section in selected:
        entries = RESULT_DATA_GUIDE.get(section, [])
        if not entries:
            continue
        lines.append(f"## {section}")
        lines.append("")
        for entry in entries:
            lines.append(f"### {entry['artifact']}")
            lines.append(f"- Role: {entry['role']}")
            lines.append(f"- What it shows: {entry['what']}")
            lines.append(f"- What to watch: {entry['watch']}")
            lines.append(f"- How to interpret: {entry['interpret']}")
            lines.append(f"- Compare with: {entry['compare']}")
            lines.append(f"- Possible insight: {entry['insight']}")
            lines.append("")
    return chr(10).join(lines).strip() + chr(10)


def display_result_data_guide(sections: str | list[str] | None = None) -> None:
    """결과 파일을 어떻게 읽을지 Markdown으로 표시합니다."""
    display(Markdown(result_data_guide_markdown(sections)))


def export_result_data_guide() -> Path:
    return write_markdown(ARTIFACT_DIR / "result_data_guide.md", result_data_guide_markdown())


def serializable_run(run: dict) -> dict:
    return {
        "summary": run.get("summary", {}),
        "history": run.get("history", []),
        "samples": run.get("samples", []),
        "config": run.get("config", {}),
    }


def config_summary_from_config(config: dict, reference_config: dict = LEARNED_GELU_CONFIG) -> str:
    changed = []
    ignored = {"name"}
    for key in sorted(config.keys()):
        if key in ignored or key not in reference_config:
            continue
        if config[key] != reference_config[key]:
            changed.append(f"{key}={config[key]}")
    return ", ".join(changed) if changed else "same as learned_gelu"


def reference_summary_for_runs(runs: list[dict], reference_name: str | None = None) -> tuple[dict | None, bool]:
    """REFERENCE_RUN_NAME과 일치하는 run summary를 찾고, 없으면 첫 run을 반환합니다."""
    if not runs:
        return None, False
    reference_name = reference_name or globals().get("REFERENCE_RUN_NAME", runs[0]["summary"]["name"])
    for run in runs:
        if run["summary"]["name"] == reference_name:
            return run["summary"], True
    return runs[0]["summary"], False


def reference_label_for_runs(runs: list[dict]) -> str:
    """표 제목에 표시할 reference run 정보를 만듭니다."""
    summary, exact = reference_summary_for_runs(runs)
    if summary is None:
        return "Reference run: none"
    if exact:
        return f"Reference run: {summary['name']}"
    requested = globals().get("REFERENCE_RUN_NAME", "")
    return f"Reference run: {summary['name']} (requested {requested} not found)"


def final_history(run: dict) -> dict:
    history = run.get("history", [])
    return history[-1] if history else {}


def lm_config_manifest_rows(items: list[dict]) -> list[dict]:
    rows = []
    for item in items:
        config = item.get("config", item)
        rows.append({
            "run": config.get("name", ""),
            "config_summary": config_summary_from_config(config),
            "position_encoding": config.get("position_encoding"),
            "activation": config.get("activation"),
            "optimizer": config.get("optimizer"),
            "scheduler": config.get("scheduler"),
            "learning_rate": config.get("learning_rate"),
            "tokenizer_vocab_size": config.get("tokenizer_vocab_size"),
            "tokenizer_train_chars": config.get("tokenizer_train_chars"),
            "lm_train_chars": config.get("lm_train_chars"),
            "lm_val_chars": config.get("lm_val_chars"),
            "context_length": config.get("context_length"),
            "batch_size": config.get("batch_size"),
            "max_steps": config.get("max_steps"),
            "estimated_params": estimate_model_parameters(config),
        })
    return rows


def lm_summary_rows(runs: list[dict]) -> list[dict]:
    rows = []
    if not runs:
        return rows
    reference, _ = reference_summary_for_runs(runs)
    reference_loss = reference["best_val_loss"] if reference else float("nan")
    for run in runs:
        s = run["summary"]
        h = final_history(run)
        final_train_loss = h.get("train_loss", s.get("final_train_loss"))
        final_val_loss = h.get("val_loss", s.get("final_val_loss"))
        rows.append({
            "run": s["name"],
            "config_summary": config_summary_from_config(run.get("config", {})),
            "position_encoding": s.get("position_encoding"),
            "activation": s.get("activation"),
            "optimizer": s.get("optimizer"),
            "scheduler": s.get("scheduler"),
            "params": s.get("total_params"),
            "train_tokens": s.get("train_tokens"),
            "val_tokens": s.get("val_tokens"),
            "tokens_seen": s.get("tokens_seen"),
            "approx_train_passes": s.get("approx_train_passes", s.get("approx_passes_over_train_tokens")),
            "best_val_loss": s.get("best_val_loss"),
            "final_train_loss": final_train_loss,
            "final_val_loss": final_val_loss,
            "best_perplexity": perplexity(s.get("best_val_loss")),
            "generalization_gap": None if final_val_loss is None or final_train_loss is None else final_val_loss - final_train_loss,
            "runtime": s.get("runtime"),
            "runtime_seconds": s.get("runtime_sec"),
            "tokens_per_sec": s.get("tokens_per_sec"),
            "reference_run": reference.get("name") if reference else "",
            "reference_delta": None if s.get("best_val_loss") is None else s.get("best_val_loss") - reference_loss,
        })
    return rows


def lm_reference_comparison_rows(runs: list[dict]) -> list[dict]:
    reference, _ = reference_summary_for_runs(runs)
    if not reference:
        return []
    reference_loss = reference["best_val_loss"]
    rows = []
    for row in lm_summary_rows(runs):
        delta = row.get("reference_delta")
        rows.append({
            "run": row["run"],
            "reference_run": reference["name"],
            "best_val_loss": row.get("best_val_loss"),
            "reference_best_val_loss": reference_loss,
            "reference_delta": delta,
            "relative_delta_pct": None if not reference_loss else 100.0 * delta / reference_loss,
            "params": row.get("params"),
            "runtime_seconds": row.get("runtime_seconds"),
            "tokens_per_sec": row.get("tokens_per_sec"),
        })
    return rows


def display_lm_summary(runs: list[dict]) -> None:
    rows = []
    for row in lm_summary_rows(runs):
        rows.append({
            "run": row["run"],
            "config": row["config_summary"],
            "params": f"{row['params']:,}" if row.get("params") is not None else "",
            "best_val_loss": format_float(row.get("best_val_loss")),
            "final_val_loss": format_float(row.get("final_val_loss")),
            "ppl": format_float(row.get("best_perplexity"), 1),
            "gap": format_float(row.get("generalization_gap")),
            "delta": format_float(row.get("reference_delta")),
            "runtime": row.get("runtime"),
            "tokens/sec": f"{row['tokens_per_sec']:.0f}" if row.get("tokens_per_sec") is not None else "",
        })
    display_table(
        rows,
        [("run", "run"), ("config", "config summary"), ("params", "params"), ("best_val_loss", "best val loss"), ("final_val_loss", "final val loss"), ("ppl", "best ppl"), ("gap", "gap"), ("delta", "reference delta"), ("runtime", "runtime"), ("tokens/sec", "tokens/sec")],
        title=f"LM core summary ({reference_label_for_runs(runs)})",
    )


def lm_history_rows(runs: list[dict]) -> list[dict]:
    rows = []
    for run in runs:
        for h in run.get("history", []):
            train_loss = h.get("train_loss")
            val_loss = h.get("val_loss")
            rows.append({
                "run": run["summary"]["name"],
                "step": h.get("step"),
                "train_loss": train_loss,
                "val_loss": val_loss,
                "train_perplexity": perplexity(train_loss),
                "val_perplexity": perplexity(val_loss),
                "val_train_gap": None if train_loss is None or val_loss is None else val_loss - train_loss,
                "learning_rate": h.get("lr"),
                "elapsed_seconds": h.get("elapsed_sec", h.get("elapsed_seconds")),
                "tokens_seen_so_far": h.get("tokens_seen_so_far"),
            })
    return rows


def lm_debug_metrics_rows(runs: list[dict]) -> list[dict]:
    rows = []
    for run in runs:
        for h in run.get("history", []):
            rows.append({
                "run": run["summary"]["name"],
                "step": h.get("step"),
                "train_token_acc": h.get("train_token_acc"),
                "val_token_acc": h.get("val_token_acc"),
                "batch_loss": h.get("batch_loss"),
                "learning_rate": h.get("lr"),
                "grad_norm": h.get("grad_norm"),
                "gpu_memory_mb": h.get("gpu_memory_mb"),
                "tokens_seen_so_far": h.get("tokens_seen_so_far"),
            })
    return rows


def generation_rows(runs: list[dict]) -> list[dict]:
    rows = []
    for run in runs:
        config = run.get("config", {})
        for sample in run.get("samples", []):
            decoding = sample.get("decoding", "balanced")
            dec_cfg = decoding_config(config, decoding) if "decoding_config" in globals() else {}
            rows.append({
                "run": run["summary"]["name"],
                "phase": sample.get("phase", "after"),
                "prompt": sample.get("prompt", ""),
                "decoding": decoding,
                "temperature": sample.get("temperature", dec_cfg.get("temperature")),
                "top_k": sample.get("top_k", dec_cfg.get("top_k")),
                "generated_text": sample.get("text", ""),
            })
    return rows


def build_generation_review_markdown(runs: list[dict]) -> str:
    lines = ["# LM Generation Review", ""]
    rows = generation_rows(runs)
    if not rows:
        lines.append("_No generation samples._")
        return chr(10).join(lines)
    prompts = []
    for row in rows:
        if row["prompt"] not in prompts:
            prompts.append(row["prompt"])
    for prompt in prompts:
        lines.append(f"## Prompt: `{prompt}`")
        prompt_rows = [r for r in rows if r["prompt"] == prompt]
        lines.extend(markdown_table(prompt_rows, [("run", "run"), ("phase", "phase"), ("decoding", "decoding"), ("temperature", "temperature"), ("top_k", "top_k"), ("generated_text", "generated text")]))
        lines.append("")
    return chr(10).join(lines).strip() + chr(10)


def build_lm_report_markdown(runs: list[dict]) -> str:
    lines = ["# LM Pretraining Report", ""]
    lines.append(f"- tokenizer preset: `{globals().get('TOKENIZER_PRESET', '')}`")
    lines.append(f"- tokenized data preset: `{globals().get('TOKENIZED_DATA_PRESET', '')}`")
    lines.append(f"- training preset: `{globals().get('TRAINING_PRESET', RUN_PRESET)}`")
    lines.append(f"- reference run: `{reference_summary_for_runs(runs)[0]['name'] if runs else REFERENCE_RUN_NAME}`")
    lines.append(f"- runs: `{len(runs)}`")
    lines.append("")
    lines.append("## Core Summary")
    summary_rows = []
    for row in lm_summary_rows(runs):
        summary_rows.append({
            "run": row["run"],
            "config": row["config_summary"],
            "best_val_loss": format_float(row.get("best_val_loss")),
            "final_val_loss": format_float(row.get("final_val_loss")),
            "ppl": format_float(row.get("best_perplexity"), 1),
            "gap": format_float(row.get("generalization_gap")),
            "delta": format_float(row.get("reference_delta")),
            "tokens/sec": f"{row['tokens_per_sec']:.0f}" if row.get("tokens_per_sec") is not None else "",
            "runtime": row.get("runtime"),
        })
    lines.extend(markdown_table(summary_rows, [("run", "run"), ("config", "config"), ("best_val_loss", "best val loss"), ("final_val_loss", "final val loss"), ("ppl", "ppl"), ("gap", "gap"), ("delta", "reference delta"), ("tokens/sec", "tokens/sec"), ("runtime", "runtime")]))
    lines.append("")
    if runs:
        best = min(runs, key=lambda r: r["summary"]["best_val_loss"])
        s = best["summary"]
        lines.append("## Best Run")
        lines.append(f"- `{s['name']}` reached best validation loss `{s['best_val_loss']:.3f}` with perplexity `{perplexity(s['best_val_loss']):.1f}`.")
        lines.append("")
    lines.append("## Generation Review")
    lines.append("See `lm_generation_review.md` for prompt-by-prompt samples.")
    return chr(10).join(lines).strip() + chr(10)


def export_lm_artifacts(runs: list[dict]) -> None:
    """LM 실험 결과를 JSONL, CSV, Markdown으로 저장합니다."""
    save_jsonl(ARTIFACT_DIR / "lm_runs.jsonl", [serializable_run(run) for run in runs])
    write_csv(ARTIFACT_DIR / "lm_config_manifest.csv", lm_config_manifest_rows(runs))
    write_csv(ARTIFACT_DIR / "lm_summary.csv", lm_summary_rows(runs), columns=LM_SUMMARY_COLUMNS)
    write_csv(ARTIFACT_DIR / "lm_history.csv", lm_history_rows(runs), columns=LM_HISTORY_COLUMNS)
    write_csv(ARTIFACT_DIR / "lm_debug_metrics.csv", lm_debug_metrics_rows(runs), columns=LM_DEBUG_COLUMNS)
    write_csv(ARTIFACT_DIR / "lm_reference_comparison.csv", lm_reference_comparison_rows(runs))
    write_csv(ARTIFACT_DIR / "generation_samples.csv", generation_rows(runs))
    write_markdown(ARTIFACT_DIR / "lm_generation_review.md", build_generation_review_markdown(runs))
    write_markdown(ARTIFACT_DIR / "lm_report_summary.md", build_lm_report_markdown(runs))
    export_result_data_guide()
    print("saved LM artifacts:", ARTIFACT_DIR)


def save_lm_runs(runs: list[dict]) -> Path:
    export_lm_artifacts(runs)
    return ARTIFACT_DIR / "lm_runs.jsonl"


def measured_steps(runs: list[dict]) -> list[int]:
    """history에 실제로 기록된 optimizer step만 모읍니다."""
    return sorted({int(h["step"]) for run in runs for h in run.get("history", []) if h.get("step") is not None})


def set_step_ticks(ax, runs: list[dict], max_ticks: int = 12) -> None:
    """x축 tick을 실제 평가가 수행된 step에 맞춥니다."""
    steps = measured_steps(runs)
    if not steps:
        return
    if len(steps) > max_ticks:
        indices = np.linspace(0, len(steps) - 1, max_ticks, dtype=int)
        steps = [steps[i] for i in sorted(set(indices))]
    ax.set_xticks(steps)


def plot_lm_histories(runs: list[dict], save: bool = True) -> None:
    if not runs:
        print("No runs to plot.")
        return
    fig, ax = plt.subplots(figsize=(12, 5))
    for run in runs:
        history = run["history"]
        steps = [h["step"] for h in history]
        train_losses = [h["train_loss"] for h in history]
        val_losses = [h["val_loss"] for h in history]
        ax.plot(steps, train_losses, marker="o", label=f"{run['summary']['name']} train")
        ax.plot(steps, val_losses, marker="x", linestyle="--", label=f"{run['summary']['name']} val")
    ax.set_xlabel("optimizer step")
    ax.set_ylabel("cross entropy loss")
    ax.set_title("Pretraining loss comparison")
    ax.legend()
    ax.grid(alpha=0.25)
    set_step_ticks(ax, runs)
    if save:
        save_current_figure("lm_loss.png")
    plt.show()


def plot_lm_perplexity(runs: list[dict], save: bool = True) -> None:
    if not runs:
        print("No runs to plot.")
        return
    fig, ax = plt.subplots(figsize=(12, 5))
    for run in runs:
        history = run["history"]
        steps = [h["step"] for h in history]
        val_ppl = [perplexity(h["val_loss"]) for h in history]
        ax.plot(steps, val_ppl, marker="o", label=run["summary"]["name"])
    ax.set_xlabel("optimizer step")
    ax.set_ylabel("validation perplexity")
    ax.set_title("Validation perplexity comparison")
    ax.legend()
    ax.grid(alpha=0.25)
    set_step_ticks(ax, runs)
    if save:
        save_current_figure("lm_perplexity.png")
    plt.show()


def plot_lm_gap(runs: list[dict], save: bool = True) -> None:
    if not runs:
        print("No runs to plot.")
        return
    fig, ax = plt.subplots(figsize=(12, 5))
    for run in runs:
        history = run["history"]
        steps = [h["step"] for h in history]
        gaps = [h["val_loss"] - h["train_loss"] for h in history]
        ax.plot(steps, gaps, marker="o", label=run["summary"]["name"])
    ax.axhline(0, color="black", linewidth=1)
    ax.set_xlabel("optimizer step")
    ax.set_ylabel("validation loss - train loss")
    ax.set_title("Validation-train loss gap")
    ax.legend()
    ax.grid(alpha=0.25)
    set_step_ticks(ax, runs)
    if save:
        save_current_figure("lm_val_train_gap.png")
    plt.show()


def plot_lm_debug_token_accuracy(runs: list[dict], save: bool = True) -> None:
    if not runs:
        print("No runs to plot.")
        return
    fig, ax = plt.subplots(figsize=(12, 5))
    for run in runs:
        history = run["history"]
        steps = [h["step"] for h in history]
        val_acc = [h.get("val_token_acc", float("nan")) for h in history]
        ax.plot(steps, val_acc, marker="o", label=run["summary"]["name"])
    ax.set_xlabel("optimizer step")
    ax.set_ylabel("validation token accuracy")
    ax.set_title("Debug validation next-token accuracy")
    ax.set_ylim(0, 1)
    ax.legend()
    ax.grid(alpha=0.25)
    set_step_ticks(ax, runs)
    if save:
        save_current_figure("lm_debug_token_accuracy.png")
    plt.show()


def plot_lm_token_accuracy(runs: list[dict], save: bool = True) -> None:
    """Backward-compatible alias. Saves the debug artifact name."""
    plot_lm_debug_token_accuracy(runs, save=save)


def plot_efficiency_scatter(runs: list[dict], save: bool = True) -> None:
    if not runs:
        print("No runs to plot.")
        return
    fig, ax = plt.subplots(figsize=(7, 5))
    for run in runs:
        s = run["summary"]
        ax.scatter(s["total_params"], s["best_val_loss"], s=80)
        ax.text(s["total_params"], s["best_val_loss"], " " + s["name"], va="center")
    ax.set_xlabel("parameter count")
    ax.set_ylabel("best validation loss")
    ax.set_title("Quality vs model size")
    ax.grid(alpha=0.25)
    if save:
        save_current_figure("lm_quality_vs_params.png")
    plt.show()


def plot_runtime_scatter(runs: list[dict], save: bool = True) -> None:
    if not runs:
        print("No runs to plot.")
        return
    fig, ax = plt.subplots(figsize=(7, 5))
    for run in runs:
        s = run["summary"]
        ax.scatter(s["runtime_sec"], s["best_val_loss"], s=80)
        ax.text(s["runtime_sec"], s["best_val_loss"], " " + s["name"], va="center")
    ax.set_xlabel("runtime seconds")
    ax.set_ylabel("best validation loss")
    ax.set_title("Quality vs runtime")
    ax.grid(alpha=0.25)
    if save:
        save_current_figure("lm_quality_vs_runtime.png")
    plt.show()


def plot_throughput_scatter(runs: list[dict], save: bool = True) -> None:
    if not runs:
        print("No runs to plot.")
        return
    fig, ax = plt.subplots(figsize=(7, 5))
    for run in runs:
        s = run["summary"]
        ax.scatter(s["tokens_per_sec"], s["best_val_loss"], s=80)
        ax.text(s["tokens_per_sec"], s["best_val_loss"], " " + s["name"], va="center")
    ax.set_xlabel("tokens per second")
    ax.set_ylabel("best validation loss")
    ax.set_title("Quality vs throughput")
    ax.grid(alpha=0.25)
    if save:
        save_current_figure("lm_quality_vs_throughput.png")
    plt.show()


def display_experiment_manifest(experiments: list[dict]) -> None:
    """학습 전에 run 목록과 LEARNED_GELU_CONFIG 대비 변경점을 확인하고 manifest CSV를 저장합니다."""
    rows = lm_config_manifest_rows(experiments)
    write_csv(ARTIFACT_DIR / "lm_config_manifest.csv", rows)
    display_table(rows, [("run", "run"), ("config_summary", "config summary"), ("estimated_params", "estimated params"), ("tokenizer_vocab_size", "tok vocab"), ("tokenizer_train_chars", "tok chars"), ("lm_train_chars", "lm chars"), ("max_steps", "steps"), ("context_length", "ctx"), ("batch_size", "batch"), ("learning_rate", "lr"), ("optimizer", "optimizer")], title="Experiment manifest")


DATA_LABEL_DISTRIBUTION_ROWS = export_data_label_distribution(show_table=True, save_plot=True)
display_result_data_guide("Data")
export_result_data_guide()


## 8. 토크나이저 준비와 토큰화 캐시

**목적**
- 5번 셀에서 고른 `TOKENIZER_CONFIG`와 `TOKENIZED_DATA_CONFIG`를 실제 tokenizer/token id cache로 준비합니다.
- 모델 구조/optimizer/학습 step 실험을 할 때 tokenizer와 입력 token을 고정합니다.

**수정할 곳**
- 보통은 5번 셀의 `TOKENIZER_PRESET`, `TOKENIZED_DATA_PRESET`만 수정합니다.
- `USE_PREPARED_TOKENIZER = True`: 기본 권장. 아래 모델/학습 실험은 같은 tokenizer를 공유합니다.
- tokenizer만 비교할 때는 `USE_PREPARED_TOKENIZER = False`로 바꾸고 `TOKENIZER_EXPERIMENTS` 또는 `README_TOKENIZER_EXPERIMENTS`를 실행합니다.

**출력**
- tokenizer cache 파일: `data/vocab_lab_v{vocab_size}_c{train_chars}.json`
- tokenized corpus cache: 같은 설정으로 다시 실행하면 encoding을 재사용합니다.
- 준비된 tokenizer 설정, 실제 vocab 크기, train/val token 수.

**해석**
- 모델 비교: 8번 셀을 한 번 실행한 뒤 `USE_PREPARED_TOKENIZER=True`로 유지하고 `POSITION_EXPERIMENTS`, `LR_EXPERIMENTS` 등을 실행합니다.
- 학습 규모 비교: `TRAINING_PRESET`만 바꾸면 tokenizer와 tokenized data는 그대로 유지됩니다.
- 데이터 크기 비교: `TOKENIZED_DATA_PRESET` 또는 `TOKENIZED_DATA_EXPERIMENTS`만 바꿔 실행합니다.
- tokenizer 비교: token 단위 자체가 바뀌므로 모델 구조/optimizer 실험과 분리해서 해석합니다.


In [ ]:
TOKENIZER_CACHE = {}
TOKENIZED_CORPUS_CACHE = {}

# TOKENIZER_CONFIG와 TOKENIZED_DATA_CONFIG는 5번 설정 셀에서 독립적으로 정합니다.
# 모델/학습 실험을 할 때는 이 tokenizer를 고정해서 쓰는 것을 권장합니다.
USE_PREPARED_TOKENIZER = True
PREPARED_TOKENIZER = None
PREPARED_TOKENIZER_CONFIG = None
PREPARED_TOKENIZED_DATA = None
PREPARED_TOKENIZED_DATA_CONFIG = None


def train_or_load_tokenizer(vocab_size: int, corpus: str, train_chars: int) -> BPETokenizer:
    """BPE tokenizer를 학습하거나 data/ 캐시에서 재사용합니다."""
    key = (vocab_size, train_chars)
    if key in TOKENIZER_CACHE:
        return TOKENIZER_CACHE[key]

    cache_path = DATA_DIR / f"vocab_lab_v{vocab_size}_c{train_chars}.json"
    tokenizer = BPETokenizer(vocab_size=vocab_size)
    if cache_path.exists():
        tokenizer.load(cache_path)
        print("Loaded tokenizer:", cache_path)
    else:
        train_text = corpus[:train_chars]
        if not train_text.strip():
            raise ValueError("Tokenizer 학습 corpus가 비어 있습니다. 데이터 준비 셀을 먼저 확인하세요.")
        print(f"Training tokenizer: vocab_size={vocab_size}, chars={len(train_text):,}")
        started = time.time()
        tokenizer.train(train_text)
        tokenizer.save(cache_path)
        print("Saved tokenizer:", cache_path, "elapsed=", format_seconds(time.time() - started))

    TOKENIZER_CACHE[key] = tokenizer
    return tokenizer


def prepare_tokenizer(config: dict | None = None) -> BPETokenizer:
    """실험 전에 tokenizer를 한 번 준비합니다.

    USE_PREPARED_TOKENIZER=True이면 run_many(...)의 모든 실험이 이 tokenizer를 공유합니다.
    tokenizer 실험을 하고 싶을 때는 USE_PREPARED_TOKENIZER=False로 두고,
    실험 config에 tokenizer 값을 명시해 개별 tokenizer를 사용합니다.
    """
    global PREPARED_TOKENIZER, PREPARED_TOKENIZER_CONFIG
    config = TOKENIZER_CONFIG if config is None else config
    PREPARED_TOKENIZER_CONFIG = {
        "tokenizer_vocab_size": config["tokenizer_vocab_size"],
        "tokenizer_train_chars": config["tokenizer_train_chars"],
    }
    PREPARED_TOKENIZER = train_or_load_tokenizer(
        vocab_size=PREPARED_TOKENIZER_CONFIG["tokenizer_vocab_size"],
        corpus=train_corpus,
        train_chars=PREPARED_TOKENIZER_CONFIG["tokenizer_train_chars"],
    )
    print("Prepared tokenizer:", PREPARED_TOKENIZER_CONFIG)
    print("Actual vocab size:", len(PREPARED_TOKENIZER.id_to_token))
    return PREPARED_TOKENIZER


def get_tokenizer_for_experiment(config: dict) -> tuple[BPETokenizer, dict]:
    """실험에 사용할 tokenizer와 tokenizer 설정을 반환합니다."""
    if USE_PREPARED_TOKENIZER:
        tokenizer = PREPARED_TOKENIZER if PREPARED_TOKENIZER is not None else prepare_tokenizer(TOKENIZER_CONFIG)
        tokenizer_config = dict(PREPARED_TOKENIZER_CONFIG)
        return tokenizer, tokenizer_config

    tokenizer_config = {
        "tokenizer_vocab_size": config["tokenizer_vocab_size"],
        "tokenizer_train_chars": config["tokenizer_train_chars"],
    }
    tokenizer = train_or_load_tokenizer(
        vocab_size=tokenizer_config["tokenizer_vocab_size"],
        corpus=train_corpus,
        train_chars=tokenizer_config["tokenizer_train_chars"],
    )
    return tokenizer, tokenizer_config


def encode_corpus(tokenizer: BPETokenizer, text: str, char_limit: int, add_bos_eos_per_line: bool = False) -> list[int]:
    """텍스트 corpus를 LM 학습용 token id 리스트로 변환합니다."""
    text = text[:char_limit]
    if add_bos_eos_per_line:
        ids = []
        for line in text.splitlines():
            line = line.strip()
            if line:
                ids.extend(tokenizer.encode(line, add_bos_eos=True))
        return ids
    return tokenizer.encode(text)


def tokenization_config_from_experiment(config: dict, tokenizer_config: dict) -> dict:
    """token id 캐시 key로 사용할 설정만 분리합니다."""
    return {
        "tokenizer_vocab_size": tokenizer_config["tokenizer_vocab_size"],
        "tokenizer_train_chars": tokenizer_config["tokenizer_train_chars"],
        "lm_train_chars": config["lm_train_chars"],
        "lm_val_chars": config["lm_val_chars"],
        "add_bos_eos_per_line": bool(config.get("add_bos_eos_per_line", False)),
    }


def prepare_tokenized_corpus(config: dict | None = None, tokenizer: BPETokenizer | None = None, tokenizer_config: dict | None = None):
    """train/val 텍스트를 token id로 변환하고 같은 설정이면 메모리 cache를 재사용합니다."""
    global PREPARED_TOKENIZED_DATA, PREPARED_TOKENIZED_DATA_CONFIG
    config = LEARNED_GELU_CONFIG if config is None else config
    tokenizer = PREPARED_TOKENIZER if tokenizer is None else tokenizer
    if tokenizer is None:
        tokenizer = prepare_tokenizer(TOKENIZER_CONFIG)

    if tokenizer_config is None:
        if USE_PREPARED_TOKENIZER and PREPARED_TOKENIZER_CONFIG is not None:
            tokenizer_config = PREPARED_TOKENIZER_CONFIG
        else:
            tokenizer_config = {
                "tokenizer_vocab_size": config["tokenizer_vocab_size"],
                "tokenizer_train_chars": config["tokenizer_train_chars"],
            }

    token_config = tokenization_config_from_experiment(config, tokenizer_config)
    cache_key = tuple(sorted(token_config.items()))
    if cache_key not in TOKENIZED_CORPUS_CACHE:
        print("Encoding corpus with tokenizer config:", token_config)
        started = time.time()
        train_ids = encode_corpus(tokenizer, train_corpus, config["lm_train_chars"], token_config["add_bos_eos_per_line"])
        val_ids = encode_corpus(tokenizer, val_corpus, config["lm_val_chars"], token_config["add_bos_eos_per_line"])
        stats = {"train_tokens": len(train_ids), "val_tokens": len(val_ids)}
        TOKENIZED_CORPUS_CACHE[cache_key] = (train_ids, val_ids, stats)
        print("Encoded tokens:", stats, "elapsed=", format_seconds(time.time() - started))
    else:
        train_ids, val_ids, stats = TOKENIZED_CORPUS_CACHE[cache_key]
        print("Reused tokenized corpus:", stats)

    PREPARED_TOKENIZED_DATA_CONFIG = token_config
    PREPARED_TOKENIZED_DATA = TOKENIZED_CORPUS_CACHE[cache_key]
    return PREPARED_TOKENIZED_DATA


# 이 셀을 실행하면 tokenizer 학습/로드와 train/val token id 변환까지 먼저 준비됩니다.
# 모델 비교 실험에서는 이 결과를 계속 재사용합니다.
prepare_tokenizer(TOKENIZER_CONFIG)
prepare_tokenized_corpus(TOKENIZED_DATA_CONFIG, PREPARED_TOKENIZER, PREPARED_TOKENIZER_CONFIG)


## 8.1 DataLoader 준비

**목적**
- 8번 셀에서 준비한 token id를 학습 batch로 묶습니다.
- tokenizer는 고정하되 `context_length`, `batch_size`, `stride`는 실험마다 바꿀 수 있게 둡니다.

**수정할 곳**
- 보통 수정하지 않습니다. batch 크기나 context 길이는 3번 설정 셀 또는 실험 리스트에서 바꿉니다.

**출력**
- 각 실험 실행 시 train/val batch 수와 token 수가 기록됩니다.

**해석**
- token 수는 tokenizer와 데이터 범위의 영향입니다.
- batch 수는 token 수뿐 아니라 `context_length`, `batch_size`, `stride`의 영향을 받습니다.


In [ ]:
def build_lm_loaders(config: dict, tokenizer: BPETokenizer):
    """실험 설정으로 train/validation DataLoader와 token 통계를 만듭니다."""
    context_length = config["context_length"]
    train_ids, val_ids, token_stats = prepare_tokenized_corpus(config, tokenizer)

    if len(train_ids) <= context_length + 1:
        raise ValueError(f"train token 수가 너무 적습니다: {len(train_ids)} tokens")
    if len(val_ids) <= context_length + 1:
        # validation token이 너무 적을 때만 train 뒷부분을 임시 validation으로 분리합니다.
        split = max(context_length + 2, int(len(train_ids) * 0.1))
        val_ids = train_ids[-split:]
        train_ids = train_ids[:-split]

    stride = config.get("stride") or context_length
    train_loader = create_dataloader(
        train_ids,
        context_length=context_length,
        batch_size=config["batch_size"],
        stride=stride,
        drop_last=True,
        shuffle=True,
        num_workers=0,
    )
    val_loader = create_dataloader(
        val_ids,
        context_length=context_length,
        batch_size=config["batch_size"],
        stride=stride,
        drop_last=False,
        shuffle=False,
        num_workers=0,
    )
    token_stats = {
        "train_tokens": len(train_ids),
        "val_tokens": len(val_ids),
        "train_batches": len(train_loader),
        "val_batches": len(val_loader),
    }
    return train_loader, val_loader, token_stats


## 9. Optimizer, scheduler, 학습 루프

**목적**
- 모델을 만들고 `max_steps` 기준으로 사전학습을 실행합니다.

**수정할 곳**
- `optimizer`, `learning_rate`, `scheduler`, `warmup_steps`, `grad_clip`, `grad_accum_steps`, `amp`

**출력**
- step별 train/val loss history
- best validation loss, runtime, tokens/sec

**해석**
- loss가 거의 내려가지 않으면 learning rate나 데이터 크기를 먼저 확인합니다.
- train loss만 내려가고 val loss가 오르면 과적합 가능성이 큽니다.
- 큰 모델이 작은 모델보다 못하면 step 수, lr, 데이터량이 부족할 수 있습니다.


In [ ]:
def model_config_from_experiment(config: dict) -> dict:
    """실험 설정 dict에서 LabGPTModel이 필요한 값만 추립니다."""
    return {
        "vocab_size": config.get("actual_vocab_size", config["tokenizer_vocab_size"]),
        "context_length": config["context_length"],
        "emb_dim": config["emb_dim"],
        "n_heads": config["n_heads"],
        "n_layers": config["n_layers"],
        "drop_rate": config["drop_rate"],
        "qkv_bias": config["qkv_bias"],
        "ffn_mult": config.get("ffn_mult", 4),
        "position_encoding": config.get("position_encoding", "learned"),
        "activation": config.get("activation", "gelu"),
    }


def build_optimizer(model: nn.Module, config: dict):
    """실험 설정에 따라 optimizer를 만듭니다."""
    name = config.get("optimizer", "adamw").lower()
    lr = config["learning_rate"]
    wd = config.get("weight_decay", 0.0)
    params = [p for p in model.parameters() if p.requires_grad]
    if name == "adamw":
        return torch.optim.AdamW(params, lr=lr, weight_decay=wd)
    if name == "adam":
        return torch.optim.Adam(params, lr=lr, weight_decay=wd)
    if name == "sgd":
        return torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=wd)
    raise ValueError(f"Unknown optimizer: {name}")


def build_scheduler(optimizer, config: dict):
    """none 또는 warmup+cosine learning rate schedule을 반환합니다."""
    name = config.get("scheduler", "none").lower()
    if name == "none":
        return None
    if name == "cosine_warmup":
        warmup_steps = max(1, int(config.get("warmup_steps", 10)))
        max_steps = max(warmup_steps + 1, int(config["max_steps"]))

        def lr_lambda(step: int):
            # warmup 동안 lr을 선형 증가시키고 이후 cosine으로 천천히 낮춥니다.
            if step < warmup_steps:
                return max(1e-8, (step + 1) / warmup_steps)
            progress = (step - warmup_steps) / max(1, max_steps - warmup_steps)
            return 0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))

        return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    raise ValueError(f"Unknown scheduler: {name}")


def next_batch(iterator, loader):
    """DataLoader iterator가 끝나면 다시 처음부터 이어서 batch를 꺼냅니다."""
    try:
        return next(iterator), iterator
    except StopIteration:
        iterator = iter(loader)
        return next(iterator), iterator


def calc_lm_metrics_loader(loader, model: nn.Module, device, num_batches: int = 5) -> dict:
    """일부 batch에서 loss와 next-token top-1 accuracy를 계산합니다."""
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    correct_tokens = 0
    with torch.no_grad():
        for batch_idx, (input_batch, target_batch) in enumerate(loader):
            if batch_idx >= num_batches:
                break
            input_batch = input_batch.to(device)
            target_batch = target_batch.to(device)
            loss, logits = model(input_batch, targets=target_batch)
            token_count = target_batch.numel()
            total_loss += float(loss.detach().cpu()) * token_count
            total_tokens += token_count
            pred = logits.argmax(dim=-1)
            correct_tokens += int((pred == target_batch).sum().detach().cpu().item())
    return {
        "loss": total_loss / max(1, total_tokens),
        "token_acc": correct_tokens / max(1, total_tokens),
    }


def evaluate_lm(model: nn.Module, train_loader, val_loader, config: dict) -> dict:
    """일부 batch만 사용해 train/validation loss와 token accuracy를 평가합니다."""
    eval_batches = config.get("eval_batches", 5)
    train_metrics = calc_lm_metrics_loader(train_loader, model, device, num_batches=eval_batches)
    val_metrics = calc_lm_metrics_loader(val_loader, model, device, num_batches=eval_batches)
    return {
        "train_loss": float(train_metrics["loss"]),
        "val_loss": float(val_metrics["loss"]),
        "train_token_acc": float(train_metrics["token_acc"]),
        "val_token_acc": float(val_metrics["token_acc"]),
    }


def grad_norm_l2(parameters) -> float:
    """현재 gradient의 L2 norm을 계산합니다. gradient가 없으면 0을 반환합니다."""
    total = 0.0
    for p in parameters:
        if p.grad is not None:
            param_norm = p.grad.detach().data.norm(2).item()
            total += param_norm ** 2
    return float(total ** 0.5)


def current_gpu_memory_mb() -> float | None:
    """CUDA 사용 시 peak allocated memory를 MB 단위로 반환합니다."""
    if device.type != "cuda":
        return None
    return float(torch.cuda.max_memory_allocated() / (1024 ** 2))


def train_lm_model(config: dict, tokenizer: BPETokenizer, train_loader, val_loader, token_stats: dict):
    """하나의 LM 실험을 학습하고 history와 summary를 반환합니다."""
    set_seed(config.get("seed", GLOBAL_SEED))
    model_cfg = model_config_from_experiment(config)
    model = LabGPTModel(model_cfg).to(device)
    optimizer = build_optimizer(model, config)
    scheduler = build_scheduler(optimizer, config)

    # AMP는 GPU에서 mixed precision을 써서 속도와 메모리를 아끼는 옵션입니다.
    use_amp = bool(config.get("amp", True)) and device.type == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    grad_accum_steps = max(1, int(config.get("grad_accum_steps", 1)))
    max_steps = int(config["max_steps"])
    eval_every = int(config.get("eval_every", 50))
    grad_clip = config.get("grad_clip", None)
    tokens_per_step = config["batch_size"] * config["context_length"] * grad_accum_steps
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    history = []
    started = time.time()
    train_iter = iter(train_loader)

    initial_metrics = evaluate_lm(model, train_loader, val_loader, config)
    history.append({
        "step": 0,
        "train_loss": initial_metrics["train_loss"],
        "val_loss": initial_metrics["val_loss"],
        "train_token_acc": initial_metrics["train_token_acc"],
        "val_token_acc": initial_metrics["val_token_acc"],
        "lr": optimizer.param_groups[0]["lr"],
        "batch_loss": None,
        "grad_norm": None,
        "gpu_memory_mb": current_gpu_memory_mb(),
        "tokens_seen_so_far": 0,
        "elapsed_sec": 0.0,
    })
    print(f"[{config['name']}] step 0000 | train {initial_metrics['train_loss']:.3f} | val {initial_metrics['val_loss']:.3f} | val token acc {initial_metrics['val_token_acc']:.3f}")

    for step in range(1, max_steps + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        micro_losses = []

        # grad_accum_steps > 1이면 여러 micro batch의 gradient를 누적한 뒤 한 번 optimizer step을 진행합니다.
        for _ in range(grad_accum_steps):
            (input_batch, target_batch), train_iter = next_batch(train_iter, train_loader)
            input_batch = input_batch.to(device)
            target_batch = target_batch.to(device)

            with torch.autocast(device_type=device.type, enabled=use_amp):
                loss, _ = model(input_batch, targets=target_batch)
                scaled_loss = loss / grad_accum_steps

            scaler.scale(scaled_loss).backward()
            micro_losses.append(float(loss.detach().cpu()))

        scaler.unscale_(optimizer)
        grad_norm = grad_norm_l2(model.parameters())
        if grad_clip is not None and grad_clip > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

        scaler.step(optimizer)
        scaler.update()
        if scheduler is not None:
            scheduler.step()

        if step % eval_every == 0 or step == max_steps:
            metrics = evaluate_lm(model, train_loader, val_loader, config)
            elapsed = time.time() - started
            history.append({
                "step": step,
                "train_loss": metrics["train_loss"],
                "val_loss": metrics["val_loss"],
                "train_token_acc": metrics["train_token_acc"],
                "val_token_acc": metrics["val_token_acc"],
                "lr": optimizer.param_groups[0]["lr"],
                "batch_loss": statistics.mean(micro_losses),
                "grad_norm": grad_norm,
                "gpu_memory_mb": current_gpu_memory_mb(),
                "tokens_seen_so_far": step * tokens_per_step,
                "elapsed_sec": elapsed,
            })
            print(f"[{config['name']}] step {step:04d} | train {metrics['train_loss']:.3f} | val {metrics['val_loss']:.3f} | val token acc {metrics['val_token_acc']:.3f} | lr {optimizer.param_groups[0]['lr']:.2e} | {format_seconds(elapsed)}")

    runtime_sec = time.time() - started
    if config.get("save_checkpoint", False):
        ckpt_path = ARTIFACT_DIR / f"{config['name']}_step{max_steps}.pt"
        torch.save({"model_state_dict": model.state_dict(), "config": model_cfg, "experiment": config}, ckpt_path)
        print("checkpoint saved:", ckpt_path)

    tokens_seen = max_steps * tokens_per_step
    best_val_loss = min(h["val_loss"] for h in history)
    best_val_token_acc = max(h.get("val_token_acc", 0.0) for h in history)
    final_train_loss = history[-1]["train_loss"]
    final_val_loss = history[-1]["val_loss"]
    final_val_token_acc = history[-1].get("val_token_acc", float("nan"))
    approx_train_passes = tokens_seen / max(1, token_stats.get("train_tokens", 0))

    summary = {
        "name": config["name"],
        "position_encoding": config.get("position_encoding", "learned"),
        "activation": config.get("activation", "gelu"),
        "optimizer": config.get("optimizer", "adamw"),
        "scheduler": config.get("scheduler", "none"),
        "total_params": count_parameters(model),
        "trainable_params": count_parameters(model, trainable_only=True),
        "best_val_loss": best_val_loss,
        "best_val_token_acc": best_val_token_acc,
        "final_val_token_acc": final_val_token_acc,
        "final_train_loss": final_train_loss,
        "final_val_loss": final_val_loss,
        "approx_train_passes": approx_train_passes,
        "approx_passes_over_train_tokens": approx_train_passes,
        "runtime_sec": runtime_sec,
        "runtime": format_seconds(runtime_sec),
        "tokens_seen": tokens_seen,
        "tokens_per_sec": tokens_seen / max(1e-9, runtime_sec),
        **token_stats,
    }
    return model, history, summary


## 10. 생성문 비교와 실험 실행 함수

**목적**
- 학습 전/후 모델에 같은 prompt를 넣어 생성 결과를 비교합니다.
- `run_many(...)`로 여러 실험 설정을 순서대로 학습합니다. `USE_PREPARED_TOKENIZER=True`이면 8번 셀에서 준비한 tokenizer와 token id 캐시를 재사용합니다.

**수정할 곳**
- `PROMPTS`: 비교할 시작 문장.
- `GENERATION_PRESETS`: greedy/balanced/creative decoding 값.

**출력**
- `before`와 `after` 생성문 비교표.
- `generation_samples.csv`.

**해석**
- byte-level BPE 모델은 학습 초기에 완성되지 않은 UTF-8 byte를 만들 수 있습니다. 이 노트북은 그런 경우에도 멈추지 않고 `�`로 표시합니다.
- 생성문이 좋아 보여도 val loss가 나쁘면 우연일 수 있습니다.
- 같은 prompt에서 before보다 after가 더 자연스러워지는지 확인합니다.

**실행 단위**
- `run_experiment(config)`: 실험 설정 1개를 학습합니다. `USE_PREPARED_TOKENIZER=True`이면 8번 셀의 준비된 tokenizer를 씁니다. 같은 tokenization 설정이면 token id도 재사용합니다.
- `run_many([config1, config2, ...])`: 실험 설정 여러 개를 앞에서부터 하나씩 학습합니다.
- `POSITION_EXPERIMENTS` 같은 이름은 “설정 리스트”입니다. 함수가 아니라 `run_many(...)` 안에 넣어 실행합니다.


In [ ]:
def decoding_config(config: dict, preset_name: str = "balanced") -> dict:
    """generation preset과 실험별 generation 설정을 합칩니다."""
    preset_cfg = GENERATION_PRESETS.get(preset_name, GENERATION_PRESETS["balanced"])
    return {
        "temperature": preset_cfg.get("temperature", config.get("generation_temperature", 0.8)),
        "top_k": preset_cfg.get("top_k", config.get("generation_top_k", 40)),
    }


def safe_decode_token_ids(tokenizer: BPETokenizer, token_ids: list[int], skip_special: bool = True) -> str:
    """생성 token id를 문자열로 바꿉니다.

    byte-level BPE 모델은 학습 초기에 임의 byte를 만들 수 있습니다.
    그 byte들이 UTF-8 문자로 완성되지 않으면 tokenizer.decode()가 UnicodeDecodeError를 냅니다.
    playground와 생성문 비교에서는 실험이 멈추지 않도록 잘못된 byte를 대체 문자(�)로 표시합니다.
    """
    try:
        return tokenizer.decode(token_ids, skip_special=skip_special)
    except UnicodeDecodeError:
        special_ids = {
            tokenizer.get_pad_id(),
            tokenizer.get_unk_id(),
            tokenizer.get_bos_id(),
            tokenizer.get_eos_id(),
        }

        def expand_token(token_id: int) -> list[int]:
            token = tokenizer.id_to_token.get(int(token_id))
            if token is None:
                return []
            if isinstance(token, bytes):
                return list(token)
            if isinstance(token, tuple):
                left, right = token
                return expand_token(left) + expand_token(right)
            if isinstance(token, str):
                return [] if skip_special and token_id in special_ids else list(token.encode("utf-8"))
            return []

        byte_values = []
        for token_id in token_ids:
            token_id = int(token_id)
            if skip_special and token_id in special_ids:
                continue
            byte_values.extend(expand_token(token_id))
        return bytes(byte_values).decode("utf-8", errors="replace")


def generate_text(model: nn.Module, tokenizer: BPETokenizer, prompt: str, config: dict, decoding: str = "balanced") -> str:
    """프롬프트를 token id로 바꾸고 모델이 이어 쓴 텍스트를 반환합니다."""
    model.eval()
    encoded = tokenizer.encode(prompt)
    if not encoded:
        encoded = [tokenizer.get_bos_id()]
    idx = torch.tensor(encoded, dtype=torch.long, device=device).unsqueeze(0)
    dec_cfg = decoding_config(config, decoding)
    token_ids = generate(
        model=model,
        idx=idx,
        max_new_tokens=config.get("max_new_tokens", 50),
        context_size=config["context_length"],
        temperature=dec_cfg["temperature"],
        top_k=dec_cfg["top_k"],
    )
    return safe_decode_token_ids(tokenizer, token_ids.squeeze(0).tolist()).replace(chr(10), " ")


def generate_samples(model: nn.Module, tokenizer: BPETokenizer, config: dict, prompts: list[str], phase: str, decoding_names: list[str] | None = None) -> list[dict]:
    """여러 prompt와 decoding preset에 대한 생성 결과를 표 형태 데이터로 만듭니다."""
    decoding_names = decoding_names or ["balanced"]
    samples = []
    for decoding in decoding_names:
        dec_cfg = decoding_config(config, decoding)
        for prompt in prompts:
            try:
                text = generate_text(model, tokenizer, prompt, config, decoding=decoding)
            except Exception as e:
                text = f"[generation failed: {e}]"
            samples.append({
                "phase": phase,
                "decoding": decoding,
                "prompt": prompt,
                "temperature": dec_cfg.get("temperature"),
                "top_k": dec_cfg.get("top_k"),
                "text": text,
            })
    return samples


def build_fresh_lm_model(config: dict, tokenizer: BPETokenizer) -> nn.Module:
    """같은 seed/config로 학습 전 모델을 다시 만들어 before 샘플을 생성합니다."""
    set_seed(config.get("seed", GLOBAL_SEED))
    fresh_config = dict(config)
    fresh_config["actual_vocab_size"] = len(tokenizer.id_to_token)
    return LabGPTModel(model_config_from_experiment(fresh_config)).to(device)


def run_experiment(config: dict) -> dict:
    """tokenizer 선택, before 생성, 학습, after 생성, 결과 저장을 한 번에 수행합니다."""
    config = dict(config)
    print("=" * 80)
    print("RUN:", config["name"])
    tokenizer, tokenizer_config = get_tokenizer_for_experiment(config)
    # 모델의 vocab_size는 실제 tokenizer 크기와 맞춰야 합니다.
    config.update(tokenizer_config)
    config["actual_vocab_size"] = len(tokenizer.id_to_token)
    print("tokenizer config:", tokenizer_config)

    before_model = build_fresh_lm_model(config, tokenizer)
    before_samples = generate_samples(before_model, tokenizer, config, PROMPTS, phase="before", decoding_names=["balanced"])
    del before_model
    if device.type == "cuda":
        torch.cuda.empty_cache()

    train_loader, val_loader, token_stats = build_lm_loaders(config, tokenizer)
    print("token stats:", token_stats)
    model, history, summary = train_lm_model(config, tokenizer, train_loader, val_loader, token_stats)
    after_samples = generate_samples(model, tokenizer, config, PROMPTS, phase="after", decoding_names=list(GENERATION_PRESETS.keys()))
    samples = before_samples + after_samples

    run = {
        "config": dict(config),
        "tokenizer": tokenizer,
        "model": model,
        "history": history,
        "summary": summary,
        "samples": samples,
    }
    RUNS.append(run)
    save_lm_runs(RUNS)
    return run


def run_many(experiments: list[dict]) -> list[dict]:
    """실험 설정 리스트를 받아 앞에서부터 하나씩 학습합니다.

    예: run_many(POSITION_EXPERIMENTS)는 위치 임베딩 실험 3개를 순서대로 실행합니다.
    """
    display_experiment_manifest(experiments)
    new_runs = []
    for config in experiments:
        run = run_experiment(config)
        new_runs.append(run)
        display_lm_summary(RUNS)
        plot_lm_histories(RUNS)
        plot_lm_debug_token_accuracy(RUNS)
    return new_runs


def display_generation_comparison(runs: list[dict]) -> None:
    rows = generation_rows(runs)
    display_table(rows, [("run", "run"), ("phase", "phase"), ("decoding", "decoding"), ("temperature", "temperature"), ("top_k", "top_k"), ("prompt", "prompt"), ("generated_text", "generated text")], title="Generation comparison")


def select_best_lm_run(runs: list[dict] | None = None) -> dict | None:
    """validation loss가 가장 낮은 LM run을 반환합니다."""
    runs = RUNS if runs is None else runs
    if not runs:
        return None
    return min(runs, key=lambda r: r["summary"]["best_val_loss"])


## 11. 사전학습 실험 실행

**목적**
- `EXPERIMENTS`에 정의한 사전학습 실험 목록을 실행합니다.

**수정할 곳**
- `learned_gelu`, `sinusoidal_position` 실행: `new_runs = run_many(EXPERIMENTS)`의 주석을 해제합니다.
- 예시 목록 실행: `new_runs = run_many(POSITION_EXPERIMENTS)`처럼 원하는 리스트 이름을 넣습니다.

**출력**
- 실험 manifest.
- run별 학습 로그.
- 중간 요약표와 loss 그래프.

**해석**
- `USE_PREPARED_TOKENIZER=True`에서는 8번 셀에서 준비한 tokenizer를 모든 실험이 공유합니다. 따라서 모델/학습 설정만 공정하게 비교하기 쉽습니다.
- `run_many(...)`는 괄호 안 리스트의 실험을 동시에 돌리지 않고, 위에서 아래 순서대로 하나씩 돌립니다.
- manifest에서 의도한 변인만 바뀌었는지 먼저 확인합니다.
- `SMOKE`가 끝까지 돌면 `LIGHT` 이상으로 확장합니다.

**복사해서 쓰는 예시**
```python
# learned_gelu와 sinusoidal_position 실행. 8번 셀의 tokenizer를 공유합니다.
new_runs = run_many(EXPERIMENTS)

# 위치 임베딩 비교 실행
new_runs = run_many(POSITION_EXPERIMENTS)

# learning rate 비교 실행
new_runs = run_many(LR_EXPERIMENTS)

# 실험 하나만 실행
one_run = run_experiment(make_experiment("lr_1e-4", {"learning_rate": 1e-4}))
```


In [ ]:
# 실행 전, 어떤 실험들이 어떤 차이를 갖는지 표로 확인합니다.
display_experiment_manifest(EXPERIMENTS)

# 아래 중 하나만 골라 주석을 해제해서 실행하세요.
# run_many(...)는 리스트 안의 실험 설정을 순서대로 하나씩 학습합니다.
# USE_PREPARED_TOKENIZER=True에서는 8번 셀에서 준비한 tokenizer를 모든 실험이 공유합니다.

# 1) EXPERIMENTS 실행: learned_gelu + sinusoidal_position
# new_runs = run_many(EXPERIMENTS)

# 2) 예시 목록 실행: 13번 셀의 예시 리스트 중 하나를 넣습니다.
# new_runs = run_many(POSITION_EXPERIMENTS)
# new_runs = run_many(LR_EXPERIMENTS)
# new_runs = run_many(ACTIVATION_EXPERIMENTS)

# 3) 실험 하나만 바로 실행하고 싶을 때
# one_run = run_experiment(make_experiment("my_single_run", {"learning_rate": 1e-4}))

print("준비된 EXPERIMENTS 수:", len(EXPERIMENTS))
print("EXPERIMENTS 실행 명령: new_runs = run_many(EXPERIMENTS)")
print("예시 실행 명령: new_runs = run_many(POSITION_EXPERIMENTS)")


## 12. LM Pretraining Results

**목적**
- 실행된 LM 실험을 핵심 결과, 학습 곡선, 생성문, 디버깅 지표로 나누어 확인합니다.

**수정할 곳**
- 보통 수정하지 않습니다.
- 표시 순서를 바꾸고 싶으면 아래 코드 셀의 함수 호출 순서를 바꿉니다.

**출력**
- 핵심 CSV: `lm_config_manifest.csv`, `lm_summary.csv`, `lm_history.csv`, `lm_reference_comparison.csv`, `generation_samples.csv`.
- 핵심 그래프: `lm_loss.png`, `lm_perplexity.png`, `lm_val_train_gap.png`, `lm_quality_vs_params.png`, `lm_quality_vs_runtime.png`, `lm_quality_vs_throughput.png`.
- 분석 파일: `lm_debug_metrics.csv`, `lm_debug_token_accuracy.png`, `lm_generation_review.md`, `lm_report_summary.md`.

**해석**
- 먼저 `lm_summary.csv`에서 validation loss와 perplexity를 보고, 그 다음 loss/gap 그래프로 학습 안정성을 확인합니다.
- 생성문은 정성 평가입니다. 같은 prompt와 decoding 설정을 고정하고 loss/perplexity와 함께 읽습니다.
- token accuracy, gradient norm, GPU memory는 핵심 품질 지표가 아니라 디버깅 지표입니다.


In [ ]:
display_result_data_guide("LM Pretraining")
display_lm_summary(RUNS)
plot_lm_histories(RUNS)
plot_lm_perplexity(RUNS)
plot_lm_gap(RUNS)
plot_efficiency_scatter(RUNS)
plot_runtime_scatter(RUNS)
plot_throughput_scatter(RUNS)
plot_lm_debug_token_accuracy(RUNS)
display_generation_comparison(RUNS)
export_lm_artifacts(RUNS)


## 12.1 직접 입력 Playground

**목적**
- 학습 전/후 모델에 직접 문장을 넣어 LLM처럼 생성 결과를 확인합니다.

**수정할 곳**
- `PLAYGROUND_RUN_INDEX`: 사용할 run 번호. 첫 번째 run은 `0`, 두 번째 run은 `1`입니다.
- `PLAYGROUND_PHASE`: `before`는 학습 전 모델, `after`는 학습 후 모델입니다.
- `PLAYGROUND_PROMPT`: 입력 문장입니다.
- `PLAYGROUND_DECODING`: `greedy`, `balanced`, `creative` 중 선택합니다.

**출력**
- 선택한 모델과 decoding 설정.
- 생성 텍스트.

**해석**
- 학습 초반 모델은 깨진 byte를 만들 수 있습니다. 이 경우 `�` 문자가 보일 수 있으며, 실험 실패가 아니라 아직 언어 패턴을 충분히 못 배웠다는 신호입니다.
- greedy는 안정적이지만 단조로울 수 있습니다.
- creative는 다양하지만 품질이 흔들릴 수 있습니다.

**자주 쓰는 패턴**
```python
# 첫 번째 run의 학습 후 모델로 보기
PLAYGROUND_RUN_INDEX = 0
PLAYGROUND_PHASE = "after"
PLAYGROUND_PROMPT = "이 영화는"
PLAYGROUND_DECODING = "balanced"

# 두 번째 run의 학습 전 모델과 학습 후 모델 비교
PLAYGROUND_RUN_INDEX = 1
PLAYGROUND_PHASE = "before"  # 한 번 실행
PLAYGROUND_PHASE = "after"   # 다시 실행
```


In [ ]:
# RUNS는 실행 완료된 실험 목록입니다. RUNS[0]이 첫 번째 실험, RUNS[1]이 두 번째 실험입니다.
PLAYGROUND_RUN_INDEX = 0  # @param {type:"integer"}
# before는 학습 전 랜덤 모델, after는 학습이 끝난 모델입니다.
PLAYGROUND_PHASE = "after"  # @param ["before", "after"]
PLAYGROUND_PROMPT = "이 영화는"  # @param {type:"string"}
# greedy는 가장 확률 높은 토큰만 선택, balanced/creative는 샘플링으로 다양성을 줍니다.
PLAYGROUND_DECODING = "balanced"  # @param ["greedy", "balanced", "creative"]


def run_playground(run_index: int, phase: str, prompt: str, decoding: str) -> None:
    """선택한 run의 before/after 모델로 직접 입력 생성 결과를 표시합니다."""
    if not RUNS:
        display(Markdown("_먼저 사전학습 실험을 실행하세요._"))
        return
    if run_index < 0 or run_index >= len(RUNS):
        raise IndexError(f"run_index must be between 0 and {len(RUNS) - 1}")
    run = RUNS[run_index]
    config = run["config"]
    tokenizer = run["tokenizer"]
    if phase == "before":
        model = build_fresh_lm_model(config, tokenizer)
    else:
        model = run["model"]
    text = generate_text(model, tokenizer, prompt, config, decoding=decoding)
    if phase == "before":
        del model
        if device.type == "cuda":
            torch.cuda.empty_cache()
    display_table([{
        "run": run["summary"]["name"],
        "phase": phase,
        "decoding": decoding,
        "prompt": prompt,
        "text": text,
    }], [("run", "run"), ("phase", "phase"), ("decoding", "decoding"), ("prompt", "prompt"), ("text", "generated text")], title="Playground 결과")


run_playground(PLAYGROUND_RUN_INDEX, PLAYGROUND_PHASE, PLAYGROUND_PROMPT, PLAYGROUND_DECODING)


## 13. 실험 예시 모음

**목적**
- 변인 통제 실험을 바로 실행할 수 있는 “실험 설정 리스트”를 제공합니다.
- 배포측 README의 추가 미션 후보도 같은 방식으로 실행할 수 있게 둡니다.

**수정할 곳**
- 이 셀은 실행만 하면 예시 리스트들이 만들어집니다.
- 실제 학습은 11번 셀에서 `run_many(리스트이름)`으로 실행합니다.

**출력**
- `LR_EXPERIMENTS`, `POSITION_EXPERIMENTS`, `BATCH_SIZE_EXPERIMENTS` 같은 리스트 이름.

**해석**
- `POSITION_EXPERIMENTS`는 함수가 아니라 실험 설정 3개가 들어 있는 리스트입니다.
- `run_many(POSITION_EXPERIMENTS)`는 그 3개를 순서대로 학습합니다.
- 한 번에 너무 많이 돌리면 비교가 어려워지므로, 한 축씩 실행하는 것을 권장합니다.
- 모델/학습 실험은 `USE_PREPARED_TOKENIZER=True`로 tokenizer를 고정합니다.
- tokenizer 자체를 비교할 때만 `USE_PREPARED_TOKENIZER = False`로 바꾸고 `TOKENIZER_EXPERIMENTS`를 실행합니다.

**실행 예시**
```python
# 위치 임베딩 3종 비교
new_runs = run_many(POSITION_EXPERIMENTS)

# README 권장 batch size 후보 중 작은 것부터 일부 실행
new_runs = run_many(BATCH_SIZE_EXPERIMENTS[:2])

# tokenizer를 바꾸는 실험: 모델 설정은 유지하고 vocab 크기만 바꿔 비교
USE_PREPARED_TOKENIZER = False
new_runs = run_many(TOKENIZER_EXPERIMENTS)

# 다시 모델/학습 실험으로 돌아갈 때는 tokenizer 고정 모드로 복구
USE_PREPARED_TOKENIZER = True
prepare_tokenizer(TOKENIZER_CONFIG)
prepare_tokenized_corpus(LEARNED_GELU_CONFIG, PREPARED_TOKENIZER, PREPARED_TOKENIZER_CONFIG)
```


In [ ]:
# 아래 변수들은 모두 "실험 설정 리스트"입니다.
# 실제 실행은 11번 셀에서 run_many(LR_EXPERIMENTS)처럼 호출합니다.


# tokenizer 비교
# 주의: 이 실험은 입력 token 단위 자체가 바뀌므로 모델 구조 실험과 분리해서 해석합니다.
TOKENIZER_EXPERIMENTS = [
    make_experiment("vocab_300", {"tokenizer_vocab_size": 300, "tokenizer_train_chars": TOKENIZER_CONFIG["tokenizer_train_chars"]}),
    make_experiment("vocab_500", {"tokenizer_vocab_size": 500, "tokenizer_train_chars": TOKENIZER_CONFIG["tokenizer_train_chars"]}),
    make_experiment("vocab_1000", {"tokenizer_vocab_size": 1000, "tokenizer_train_chars": TOKENIZER_CONFIG["tokenizer_train_chars"]}),
]

README_TOKENIZER_EXPERIMENTS = [
    make_experiment("readme_vocab_300", {"tokenizer_vocab_size": 300, "tokenizer_train_chars": min(TOKENIZER_CONFIG["tokenizer_train_chars"], 500_000)}),
    make_experiment("readme_vocab_2000", {"tokenizer_vocab_size": 2_000, "tokenizer_train_chars": min(TOKENIZER_CONFIG["tokenizer_train_chars"], 500_000)}),
    make_experiment("readme_vocab_3000", {"tokenizer_vocab_size": 3_000, "tokenizer_train_chars": TOKENIZER_CONFIG["tokenizer_train_chars"]}),
]


# tokenized corpus 크기 비교: tokenizer는 고정하고 학습에 쓰는 텍스트 양만 바꿉니다.
TOKENIZED_DATA_EXPERIMENTS = [
    make_experiment("data_smoke", TOKENIZED_DATA_PRESETS["SMOKE"]),
    make_experiment("data_light", TOKENIZED_DATA_PRESETS["LIGHT"]),
    make_experiment("data_core", TOKENIZED_DATA_PRESETS["CORE"]),
]

# learning rate 비교: README 권장 후보와 동일합니다.
LR_EXPERIMENTS = [
    make_experiment("lr_1e-4", {"learning_rate": 1e-4}),
    make_experiment("lr_3e-4", {"learning_rate": 3e-4}),
    make_experiment("lr_5e-4", {"learning_rate": 5e-4}),
]

# README 권장 batch/dropout/context/depth/width 후보입니다.
BATCH_SIZE_EXPERIMENTS = [
    make_experiment("batch_2", {"batch_size": 2}),
    make_experiment("batch_4", {"batch_size": 4}),
    make_experiment("batch_8", {"batch_size": 8}),
    make_experiment("batch_16", {"batch_size": 16}),
]

DROPOUT_EXPERIMENTS = [
    make_experiment("dropout_0_0", {"drop_rate": 0.0}),
    make_experiment("dropout_0_1", {"drop_rate": 0.1}),
    make_experiment("dropout_0_2", {"drop_rate": 0.2}),
]

CONTEXT_LENGTH_EXPERIMENTS = [
    make_experiment("ctx_64", {"context_length": 64}),
    make_experiment("ctx_128", {"context_length": 128}),
]

DEPTH_EXPERIMENTS = [
    make_experiment("layers_1", {"n_layers": 1}),
    make_experiment("layers_2", {"n_layers": 2}),
    make_experiment("layers_4", {"n_layers": 4}),
]

WIDTH_EXPERIMENTS = [
    make_experiment("emb_64", {"emb_dim": 64, "n_heads": 4}),
    make_experiment("emb_128", {"emb_dim": 128, "n_heads": 4}),
    make_experiment("emb_192", {"emb_dim": 192, "n_heads": 4}),
]

# 모델 크기 비교: depth와 width를 함께 바꿔 넓은 방향을 빠르게 탐색합니다.
SIZE_EXPERIMENTS = [
    make_experiment("tiny_1layer_64d", {"emb_dim": 64, "n_heads": 4, "n_layers": 1}),
    make_experiment("base_2layer_96d", {"emb_dim": 96, "n_heads": 4, "n_layers": 2}),
    make_experiment("wide_2layer_128d", {"emb_dim": 128, "n_heads": 4, "n_layers": 2}),
    make_experiment("deep_4layer_128d", {"emb_dim": 128, "n_heads": 4, "n_layers": 4}),
]

# 위치 임베딩 비교
POSITION_EXPERIMENTS = [
    make_experiment("pos_learned", {"position_encoding": "learned"}),
    make_experiment("pos_sinusoidal", {"position_encoding": "sinusoidal"}),
    make_experiment("pos_none", {"position_encoding": "none"}),
]

# 활성화 함수 비교
ACTIVATION_EXPERIMENTS = [
    make_experiment("act_gelu", {"activation": "gelu"}),
    make_experiment("act_relu", {"activation": "relu"}),
    make_experiment("act_silu", {"activation": "silu"}),
]

# optimizer/scheduler 비교: warmup, cosine decay, weight decay 실험을 포함합니다.
OPTIMIZER_EXPERIMENTS = [
    make_experiment("adamw", {"optimizer": "adamw", "scheduler": "none", "weight_decay": 0.01}),
    make_experiment("adam", {"optimizer": "adam", "scheduler": "none", "weight_decay": 0.0}),
    make_experiment("adamw_no_decay", {"optimizer": "adamw", "scheduler": "none", "weight_decay": 0.0}),
    make_experiment("adamw_cosine", {"optimizer": "adamw", "scheduler": "cosine_warmup", "warmup_steps": 20, "weight_decay": 0.01}),
]

# seed 안정성 비교
SEED_EXPERIMENTS = [
    make_experiment("seed_42", {"seed": 42}),
    make_experiment("seed_43", {"seed": 43}),
    make_experiment("seed_44", {"seed": 44}),
]

# 파인튜닝 범위 비교는 15번 셀의 FINETUNE_SCOPE_EXPERIMENTS를 사용합니다.
print("예시 리스트:")
print("TOKENIZER_EXPERIMENTS, README_TOKENIZER_EXPERIMENTS, TOKENIZED_DATA_EXPERIMENTS")
print("LR_EXPERIMENTS, BATCH_SIZE_EXPERIMENTS, DROPOUT_EXPERIMENTS, CONTEXT_LENGTH_EXPERIMENTS")
print("DEPTH_EXPERIMENTS, WIDTH_EXPERIMENTS, SIZE_EXPERIMENTS")
print("POSITION_EXPERIMENTS, ACTIVATION_EXPERIMENTS, OPTIMIZER_EXPERIMENTS, SEED_EXPERIMENTS")
print("실행 예: 11번 셀에서 new_runs = run_many(POSITION_EXPERIMENTS)")


## 14. 파인튜닝 데이터와 freeze mode

**목적**
- 감성분류 데이터로 fine-tuning을 준비하고 classification metric을 계산합니다.

**수정할 곳**
- 보통 15번 셀의 `CLASSIFIER_ONLY_FINETUNE_CONFIG` 값을 수정합니다.

**출력**
- DataLoader
- confusion matrix
- accuracy, precision, recall, F1
- confidence 높은 오답 예시

**해석**
- `classifier_only`: 빠르고 안정적입니다.
- `last_block_plus_head`: 일부 backbone 적응을 허용합니다.
- `full_finetune`: 가장 강하지만 작은 데이터에서 과적합될 수 있습니다.


In [ ]:
def load_jsonl_rows(path: Path, limit: int | None = None, seed: int = GLOBAL_SEED) -> list[dict]:
    """JSONL 감성분류 데이터를 읽고 seed 기준으로 섞은 뒤 일부만 사용합니다."""
    rows = []
    if not path.exists():
        raise FileNotFoundError(path)
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    rng = random.Random(seed)
    rng.shuffle(rows)
    return rows[:limit] if limit is not None else rows


def build_sentiment_loaders(tokenizer: BPETokenizer, config: dict):
    """fine-tuning train/val/test DataLoader를 만듭니다."""
    train_rows = load_jsonl_rows(SENTIMENT_TRAIN_PATH, config.get("finetune_train_samples"), seed=config.get("seed", GLOBAL_SEED))
    val_rows = load_jsonl_rows(SENTIMENT_VAL_PATH, config.get("finetune_val_samples"), seed=config.get("seed", GLOBAL_SEED))
    test_rows = load_jsonl_rows(SENTIMENT_TEST_PATH, config.get("finetune_test_samples"), seed=config.get("seed", GLOBAL_SEED))

    max_length = min(config.get("finetune_max_length", config["context_length"]), config["context_length"])
    train_ds = ReviewSentimentDataset(train_rows, tokenizer, max_length=max_length)
    val_ds = ReviewSentimentDataset(val_rows, tokenizer, max_length=max_length)
    test_ds = ReviewSentimentDataset(test_rows, tokenizer, max_length=max_length)

    batch_size = config.get("finetune_batch_size", config["batch_size"])
    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=False),
        DataLoader(val_ds, batch_size=batch_size, shuffle=False, drop_last=False),
        DataLoader(test_ds, batch_size=batch_size, shuffle=False, drop_last=False),
        {"train_rows": train_rows, "val_rows": val_rows, "test_rows": test_rows},
    )


def apply_freeze_mode(model: GPTForSequenceClassification, freeze_mode: str) -> None:
    """freeze_mode에 따라 backbone의 학습 범위를 조절합니다."""
    for p in model.gpt.parameters():
        p.requires_grad = False

    if freeze_mode == "classifier_only":
        pass
    elif freeze_mode == "last_block_plus_head":
        if len(model.gpt.blocks) > 0:
            for p in model.gpt.blocks[-1].parameters():
                p.requires_grad = True
        for p in model.gpt.final_norm.parameters():
            p.requires_grad = True
    elif freeze_mode == "full_finetune":
        for p in model.gpt.parameters():
            p.requires_grad = True
    else:
        raise ValueError(f"Unknown freeze_mode: {freeze_mode}")

    for p in model.classifier.parameters():
        p.requires_grad = True


def build_finetune_optimizer(model: GPTForSequenceClassification, config: dict):
    """backbone과 classifier head에 다른 learning rate를 적용합니다."""
    head_params = []
    backbone_params = []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if name.startswith("classifier") or name.startswith("dropout"):
            head_params.append(p)
        else:
            backbone_params.append(p)

    groups = []
    if backbone_params:
        groups.append({"params": backbone_params, "lr": config.get("finetune_backbone_lr", 1e-5)})
    if head_params:
        groups.append({"params": head_params, "lr": config.get("finetune_head_lr", 1e-3)})
    return torch.optim.AdamW(groups, weight_decay=config.get("finetune_weight_decay", 0.01))


def collect_sentiment_predictions(model, loader, raw_rows: list[dict]) -> list[dict]:
    """prediction, confidence, 정답 여부를 row 단위로 수집합니다."""
    predictions = []
    model.eval()
    offset = 0
    with torch.no_grad():
        for input_ids, labels in loader:
            input_ids = input_ids.to(device)
            labels = labels.to(device).long()
            logits = model(input_ids)
            probs = torch.softmax(logits, dim=-1)
            confs, preds = torch.max(probs, dim=-1)
            batch_rows = raw_rows[offset:offset + labels.size(0)]
            offset += labels.size(0)
            prob_rows = probs.cpu().tolist()
            for row, y, p, conf, prob in zip(batch_rows, labels.cpu().tolist(), preds.cpu().tolist(), confs.cpu().tolist(), prob_rows):
                predictions.append({
                    "text": row.get("text", ""),
                    "label": int(y),
                    "pred": int(p),
                    "confidence": float(conf),
                    "prob_negative": float(prob[0]),
                    "prob_positive": float(prob[1]),
                    "correct": int(y == p),
                    "error_type": "correct" if y == p else f"true_{y}_pred_{p}",
                })
    return predictions


def classification_metrics(predictions: list[dict]) -> dict:
    """binary classification precision/recall/F1을 계산합니다."""
    tp = sum(1 for r in predictions if r["label"] == 1 and r["pred"] == 1)
    tn = sum(1 for r in predictions if r["label"] == 0 and r["pred"] == 0)
    fp = sum(1 for r in predictions if r["label"] == 0 and r["pred"] == 1)
    fn = sum(1 for r in predictions if r["label"] == 1 and r["pred"] == 0)
    precision = tp / max(1, tp + fp)
    recall = tp / max(1, tp + fn)
    f1 = 2 * precision * recall / max(1e-12, precision + recall)
    accuracy = (tp + tn) / max(1, tp + tn + fp + fn)
    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1, "tp": tp, "tn": tn, "fp": fp, "fn": fn}


def confusion_matrix_from_predictions(predictions: list[dict]) -> np.ndarray:
    matrix = np.zeros((2, 2), dtype=int)
    for row in predictions:
        y, p = row["label"], row["pred"]
        if y in (0, 1) and p in (0, 1):
            matrix[y, p] += 1
    return matrix


def normalized_confusion_matrix(matrix: np.ndarray) -> np.ndarray:
    """row 기준으로 정규화한 confusion matrix를 반환합니다."""
    row_sums = matrix.sum(axis=1, keepdims=True)
    return np.divide(matrix, np.maximum(row_sums, 1), out=np.zeros_like(matrix, dtype=float), where=row_sums != 0)


def plot_confusion_matrix(matrix: np.ndarray, title: str, save_name: str | None = None, normalize: bool = False) -> None:
    values = normalized_confusion_matrix(matrix) if normalize else matrix
    plt.figure(figsize=(4, 4))
    plt.imshow(values, cmap="Blues", vmin=0 if normalize else None, vmax=1 if normalize else None)
    plt.xticks([0, 1], ["pred 0 negative", "pred 1 positive"])
    plt.yticks([0, 1], ["true 0 negative", "true 1 positive"])
    suffix = "normalized" if normalize else "counts"
    plt.title(f"Confusion matrix ({suffix}): {title}")
    for i in range(2):
        for j in range(2):
            text = f"{values[i, j]:.2f}" if normalize else str(int(values[i, j]))
            plt.text(j, i, text, ha="center", va="center", color="black")
    plt.colorbar()
    if save_name:
        save_current_figure(save_name)
    plt.show()


## 15. Fine-tuning Functions

**목적**
- 선택한 LM run을 backbone으로 사용해 감성분류 fine-tuning을 실행합니다.
- validation loss가 가장 낮았던 epoch의 model state를 복원해 test 평가와 prediction을 생성합니다.

**수정할 곳**
- `FINETUNE_EXPERIMENTS`: 비교할 freeze mode 목록입니다.
- `finetune_head_lr`, `finetune_backbone_lr`: classifier head와 backbone learning rate입니다.

**출력**
- 핵심 CSV: `finetune_config_manifest.csv`, `finetune_summary.csv`, `finetune_history.csv`, `finetune_predictions.csv`.
- 오류 분석 CSV: `finetune_high_confidence_errors.csv`, `finetune_error_breakdown.csv`.
- 그래프: `finetune_loss.png`, `finetune_accuracy_curve.png`, `finetune_scores.png`, `finetune_prf.png`, `finetune_quality_vs_trainable_params.png`, `finetune_quality_vs_runtime.png`, `confusion_{run}.png`, `confusion_{run}_normalized.png`.

**해석**
- fine-tuning backbone은 기본적으로 best validation loss LM run을 선택합니다.
- fine-tuning test metric은 마지막 epoch가 아니라 validation loss 최저 epoch의 model state로 계산됩니다.
- `classifier_only`는 빠르고 안정적이며, `full_finetune`은 성능이 좋아질 수 있지만 과적합과 비용 증가를 함께 확인해야 합니다.


In [ ]:
CLASSIFIER_ONLY_FINETUNE_CONFIG = {
    "seed": GLOBAL_SEED,
    "freeze_mode": "classifier_only",       # classifier_only | last_block_plus_head | full_finetune
    "use_pretrained_backbone": True,
    "finetune_epochs": FINETUNE_CONFIG["finetune_epochs"],
    "finetune_train_samples": FINETUNE_CONFIG["finetune_train_samples"],
    "finetune_val_samples": FINETUNE_CONFIG["finetune_val_samples"],
    "finetune_test_samples": FINETUNE_CONFIG["finetune_test_samples"],
    "finetune_batch_size": FINETUNE_CONFIG["finetune_batch_size"],
    "finetune_max_length": FINETUNE_CONFIG["finetune_max_length"],
    "finetune_backbone_lr": 1e-5,
    "finetune_head_lr": 1e-3,
    "finetune_weight_decay": 0.01,
}


def make_finetune_experiment(name: str, overrides: dict | None = None) -> dict:
    """CLASSIFIER_ONLY_FINETUNE_CONFIG에서 필요한 값만 덮어써 fine-tuning run 설정을 만듭니다."""
    config = dict(CLASSIFIER_ONLY_FINETUNE_CONFIG)
    config["name"] = name
    if overrides:
        config.update(overrides)
    return config


FINETUNE_SCOPE_EXPERIMENTS = [
    make_finetune_experiment("ft_classifier_only", {"freeze_mode": "classifier_only"}),
    make_finetune_experiment("ft_last_block", {"freeze_mode": "last_block_plus_head", "finetune_backbone_lr": 2e-5}),
    make_finetune_experiment("ft_full_finetune", {"freeze_mode": "full_finetune", "finetune_backbone_lr": 3e-5}),
]
FINETUNE_EXPERIMENTS = FINETUNE_SCOPE_EXPERIMENTS[:1] + FINETUNE_SCOPE_EXPERIMENTS[2:]


def finetune_learning_rate(optimizer) -> float:
    """여러 param group 중 가장 큰 learning rate를 대표값으로 기록합니다."""
    return max(group.get("lr", 0.0) for group in optimizer.param_groups)


def best_state_dict_for_restore(model: nn.Module) -> dict:
    """Keep a CPU copy of the best validation-loss weights."""
    return {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}


def run_finetune_experiment(config: dict, source_run: dict | None = None) -> dict:
    """하나의 fine-tuning 실험을 실행하고 metric과 prediction을 저장합니다."""
    set_seed(config.get("seed", GLOBAL_SEED))

    if source_run is None and config.get("use_pretrained_backbone", True):
        source_run = select_best_lm_run(RUNS)

    if source_run is not None and config.get("use_pretrained_backbone", True):
        tokenizer = source_run["tokenizer"]
        backbone = copy.deepcopy(source_run["model"])
        source_lm_run = source_run["summary"]["name"]
    else:
        base_lm_config = make_experiment("scratch_for_finetune")
        tokenizer = PREPARED_TOKENIZER if USE_PREPARED_TOKENIZER and PREPARED_TOKENIZER is not None else train_or_load_tokenizer(base_lm_config["tokenizer_vocab_size"], train_corpus, base_lm_config["tokenizer_train_chars"])
        base_lm_config["actual_vocab_size"] = len(tokenizer.id_to_token)
        backbone = LabGPTModel(model_config_from_experiment(base_lm_config))
        source_lm_run = "scratch"

    clf = GPTForSequenceClassification(backbone, num_labels=2, drop_rate=0.1).to(device)
    apply_freeze_mode(clf, config["freeze_mode"])
    optimizer = build_finetune_optimizer(clf, config)

    lm_context = clf.gpt.config["context_length"]
    loader_config = {
        **LEARNED_GELU_CONFIG,
        **config,
        "context_length": lm_context,
        "batch_size": config.get("finetune_batch_size", LEARNED_GELU_CONFIG["batch_size"]),
    }
    train_loader, val_loader, test_loader, raw_rows = build_sentiment_loaders(tokenizer, loader_config)
    sample_counts = {
        "train_samples": len(raw_rows["train_rows"]),
        "val_samples": len(raw_rows["val_rows"]),
        "test_samples": len(raw_rows["test_rows"]),
    }

    history = []
    best_val_loss = float("inf")
    best_val_loss_epoch = None
    selected_state = None
    started = time.time()
    for epoch in range(1, int(config["finetune_epochs"]) + 1):
        train_loss, train_acc = train_epoch_sentiment(clf, train_loader, optimizer, device)
        val_loss, val_acc = evaluate_sentiment(clf, val_loader, device)
        elapsed = time.time() - started
        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "train_accuracy": train_acc,
            "val_accuracy": val_acc,
            "val_train_loss_gap": val_loss - train_loss,
            "learning_rate": finetune_learning_rate(optimizer),
            "elapsed_seconds": elapsed,
        })
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_loss_epoch = epoch
            selected_state = best_state_dict_for_restore(clf)
        print(f"[{config['name']}] epoch {epoch} | train acc {train_acc:.3f} | val acc {val_acc:.3f}")

    if selected_state is not None:
        clf.load_state_dict(selected_state)
        print(f"[{config['name']}] restored epoch {best_val_loss_epoch} by lowest validation loss")

    test_loss, test_acc = evaluate_sentiment(clf, test_loader, device)
    predictions = collect_sentiment_predictions(clf, test_loader, raw_rows["test_rows"])
    metrics = classification_metrics(predictions)
    matrix = confusion_matrix_from_predictions(predictions)
    high_conf_errors = sorted([r for r in predictions if not r["correct"]], key=lambda r: r["confidence"], reverse=True)[:20]
    runtime_sec = time.time() - started

    summary = {
        "name": config["name"],
        "source_lm_run": source_lm_run,
        "base_run": source_lm_run,
        "freeze_mode": config["freeze_mode"],
        "trainable_params": count_parameters(clf, trainable_only=True),
        "total_params": count_parameters(clf),
        **sample_counts,
        "epochs": int(config["finetune_epochs"]),
        "selected_epoch": best_val_loss_epoch,
        "best_val_loss_epoch": best_val_loss_epoch,
        "best_val_loss": best_val_loss,
        "final_train_loss": history[-1]["train_loss"],
        "final_val_loss": history[-1]["val_loss"],
        "best_val_accuracy": max(h["val_accuracy"] for h in history),
        "best_val_acc": max(h["val_accuracy"] for h in history),
        "final_val_accuracy": history[-1]["val_accuracy"],
        "test_accuracy": test_acc,
        "test_acc": test_acc,
        "test_loss": test_loss,
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f1": metrics["f1"],
        "fp": metrics["fp"],
        "fn": metrics["fn"],
        "runtime_sec": runtime_sec,
        "runtime": format_seconds(runtime_sec),
    }
    run = {
        "config": dict(config),
        "model": clf,
        "history": history,
        "summary": summary,
        "confusion_matrix": matrix.tolist(),
        "predictions": predictions,
        "high_conf_errors": high_conf_errors,
    }
    FINETUNE_RUNS.append(run)
    export_finetune_artifacts(FINETUNE_RUNS)
    return run


def run_finetune_many(configs: list[dict], source_run: dict | None = None) -> list[dict]:
    """여러 fine-tuning 실험을 순서대로 실행합니다."""
    if source_run is None:
        source_run = select_best_lm_run(RUNS)
    runs = []
    for config in configs:
        runs.append(run_finetune_experiment(config, source_run=source_run))
    return runs


def finetune_config_manifest_rows(items: list[dict]) -> list[dict]:
    rows = []
    for item in items:
        config = item.get("config", item)
        summary = item.get("summary", {}) if isinstance(item, dict) else {}
        rows.append({
            "run": config.get("name", summary.get("name", "")),
            "source_lm_run": summary.get("source_lm_run", ""),
            "freeze_mode": config.get("freeze_mode"),
            "finetune_head_lr": config.get("finetune_head_lr"),
            "finetune_backbone_lr": config.get("finetune_backbone_lr"),
            "finetune_epochs": config.get("finetune_epochs"),
            "finetune_train_samples": config.get("finetune_train_samples"),
            "finetune_val_samples": config.get("finetune_val_samples"),
            "finetune_test_samples": config.get("finetune_test_samples"),
            "finetune_batch_size": config.get("finetune_batch_size"),
            "selected_epoch": summary.get("selected_epoch", ""),
        })
    return rows


def finetune_summary_rows(runs: list[dict]) -> list[dict]:
    rows = []
    for run in runs:
        s = run["summary"]
        rows.append({
            "run": s["name"],
            "source_lm_run": s["source_lm_run"],
            "freeze_mode": s["freeze_mode"],
            "trainable_params": s["trainable_params"],
            "total_params": s["total_params"],
            "train_samples": s["train_samples"],
            "val_samples": s["val_samples"],
            "test_samples": s["test_samples"],
            "epochs": s["epochs"],
            "selected_epoch": s.get("selected_epoch"),
            "best_val_loss_epoch": s.get("best_val_loss_epoch"),
            "best_val_loss": s["best_val_loss"],
            "final_train_loss": s["final_train_loss"],
            "final_val_loss": s["final_val_loss"],
            "best_val_accuracy": s["best_val_accuracy"],
            "test_accuracy": s["test_accuracy"],
            "precision": s["precision"],
            "recall": s["recall"],
            "f1": s["f1"],
            "fp": s["fp"],
            "fn": s["fn"],
            "runtime": s["runtime"],
            "runtime_seconds": s["runtime_sec"],
        })
    return rows


def finetune_history_rows(runs: list[dict]) -> list[dict]:
    rows = []
    for run in runs:
        for h in run.get("history", []):
            rows.append({"run": run["summary"]["name"], **h})
    return rows


def finetune_prediction_rows(runs: list[dict]) -> list[dict]:
    rows = []
    for run in runs:
        for pred in run.get("predictions", []):
            rows.append({"run": run["summary"]["name"], **pred})
    return rows


def finetune_high_confidence_error_rows(runs: list[dict], threshold: float | None = None) -> list[dict]:
    rows = []
    for run in runs:
        errors = [r for r in run.get("predictions", []) if not r["correct"]]
        if threshold is not None:
            errors = [r for r in errors if r.get("confidence", 0.0) >= threshold]
        errors = sorted(errors, key=lambda r: r["confidence"], reverse=True)
        for row in errors:
            rows.append({"run": run["summary"]["name"], **row})
    return rows


def finetune_error_breakdown_rows(runs: list[dict]) -> list[dict]:
    rows = []
    for run in runs:
        preds = run.get("predictions", [])
        total = max(1, len(preds))
        for error_type in ["correct", "true_0_pred_1", "true_1_pred_0"]:
            count = sum(1 for r in preds if r.get("error_type") == error_type)
            rows.append({
                "run": run["summary"]["name"],
                "error_type": error_type,
                "count": count,
                "share": count / total,
            })
    return rows


def build_finetune_report_markdown(runs: list[dict]) -> str:
    lines = ["# Fine-tuning Report", ""]
    lines.append(f"- runs: `{len(runs)}`")
    lines.append("")
    lines.append("## Task Metric Summary")
    summary_rows = []
    for row in finetune_summary_rows(runs):
        summary_rows.append({
            "run": row["run"],
            "source": row["source_lm_run"],
            "freeze": row["freeze_mode"],
            "selected_epoch": row.get("selected_epoch"),
            "best_val_loss": format_float(row["best_val_loss"]),
            "val_acc": format_float(row["best_val_accuracy"]),
            "test_acc": format_float(row["test_accuracy"]),
            "precision": format_float(row["precision"]),
            "recall": format_float(row["recall"]),
            "f1": format_float(row["f1"]),
            "fp/fn": f"{row['fp']}/{row['fn']}",
            "runtime": row["runtime"],
        })
    lines.extend(markdown_table(summary_rows, [("run", "run"), ("source", "source LM"), ("freeze", "freeze"), ("selected_epoch", "selected epoch"), ("best_val_loss", "best val loss"), ("val_acc", "best val acc"), ("test_acc", "test acc"), ("precision", "precision"), ("recall", "recall"), ("f1", "F1"), ("fp/fn", "FP/FN"), ("runtime", "runtime")]))
    lines.append("")
    lines.append("## Error Breakdown")
    lines.extend(markdown_table(finetune_error_breakdown_rows(runs), [("run", "run"), ("error_type", "error type"), ("count", "count"), ("share", "share")]))
    lines.append("")
    high_errors = finetune_high_confidence_error_rows(runs)[:10]
    lines.append("## High-confidence Errors")
    lines.extend(markdown_table(high_errors, [("run", "run"), ("label", "label"), ("pred", "pred"), ("confidence", "confidence"), ("text", "text")]))
    return chr(10).join(lines).strip() + chr(10)


def export_finetune_artifacts(runs: list[dict]) -> None:
    """fine-tuning 결과를 JSONL, CSV, Markdown으로 저장합니다."""
    serializable = [{"summary": r["summary"], "history": r["history"], "config": r["config"], "confusion_matrix": r["confusion_matrix"], "high_conf_errors": r["high_conf_errors"]} for r in runs]
    save_jsonl(ARTIFACT_DIR / "finetune_runs.jsonl", serializable)
    write_csv(ARTIFACT_DIR / "finetune_config_manifest.csv", finetune_config_manifest_rows(runs))
    write_csv(ARTIFACT_DIR / "finetune_summary.csv", finetune_summary_rows(runs), columns=FINETUNE_SUMMARY_COLUMNS)
    write_csv(ARTIFACT_DIR / "finetune_history.csv", finetune_history_rows(runs))
    write_csv(ARTIFACT_DIR / "finetune_predictions.csv", finetune_prediction_rows(runs))
    write_csv(ARTIFACT_DIR / "finetune_high_confidence_errors.csv", finetune_high_confidence_error_rows(runs))
    write_csv(ARTIFACT_DIR / "finetune_error_breakdown.csv", finetune_error_breakdown_rows(runs))
    write_markdown(ARTIFACT_DIR / "finetune_report_summary.md", build_finetune_report_markdown(runs))
    export_result_data_guide()
    print("saved finetune artifacts:", ARTIFACT_DIR)


def display_finetune_summary(runs: list[dict]) -> None:
    rows = []
    for row in finetune_summary_rows(runs):
        rows.append({
            "run": row["run"],
            "source": row["source_lm_run"],
            "freeze": row["freeze_mode"],
            "trainable": f"{row['trainable_params']:,}",
            "selected_epoch": row.get("selected_epoch"),
            "best_val_loss": format_float(row["best_val_loss"]),
            "val_acc": format_float(row["best_val_accuracy"]),
            "test_acc": format_float(row["test_accuracy"]),
            "precision": format_float(row["precision"]),
            "recall": format_float(row["recall"]),
            "f1": format_float(row["f1"]),
            "fp/fn": f"{row['fp']}/{row['fn']}",
            "runtime": row["runtime"],
        })
    display_table(rows, [("run", "run"), ("source", "source LM"), ("freeze", "freeze"), ("trainable", "trainable params"), ("selected_epoch", "selected epoch"), ("best_val_loss", "best val loss"), ("val_acc", "best val acc"), ("test_acc", "test acc"), ("precision", "precision"), ("recall", "recall"), ("f1", "F1"), ("fp/fn", "FP/FN"), ("runtime", "runtime")], title="Fine-tuning core summary")


def plot_finetune_loss(runs: list[dict], save: bool = True) -> None:
    if not runs:
        print("No finetune runs to plot.")
        return
    fig, ax = plt.subplots(figsize=(10, 4))
    for run in runs:
        epochs = [h["epoch"] for h in run["history"]]
        ax.plot(epochs, [h["train_loss"] for h in run["history"]], marker="o", label=f"{run['summary']['name']} train")
        ax.plot(epochs, [h["val_loss"] for h in run["history"]], marker="x", linestyle="--", label=f"{run['summary']['name']} val")
    ax.set_xlabel("epoch")
    ax.set_ylabel("cross entropy loss")
    ax.set_title("Fine-tuning loss curve")
    ax.legend()
    ax.grid(alpha=0.25)
    if save:
        save_current_figure("finetune_loss.png")
    plt.show()


def plot_finetune_accuracy_curve(runs: list[dict], save: bool = True) -> None:
    if not runs:
        print("No finetune runs to plot.")
        return
    fig, ax = plt.subplots(figsize=(10, 4))
    for run in runs:
        epochs = [h["epoch"] for h in run["history"]]
        ax.plot(epochs, [h["train_accuracy"] for h in run["history"]], marker="o", label=f"{run['summary']['name']} train")
        ax.plot(epochs, [h["val_accuracy"] for h in run["history"]], marker="x", linestyle="--", label=f"{run['summary']['name']} val")
    ax.set_xlabel("epoch")
    ax.set_ylabel("accuracy")
    ax.set_title("Fine-tuning accuracy curve")
    ax.set_ylim(0, 1)
    ax.legend()
    ax.grid(alpha=0.25)
    if save:
        save_current_figure("finetune_accuracy_curve.png")
    plt.show()


def plot_finetune_scores(runs: list[dict], save: bool = True) -> None:
    if not runs:
        print("No finetune runs to plot.")
        return
    labels = [r["summary"]["name"] for r in runs]
    val_acc = [r["summary"]["best_val_accuracy"] for r in runs]
    test_acc = [r["summary"]["test_accuracy"] for r in runs]
    f1 = [r["summary"]["f1"] for r in runs]
    x = np.arange(len(labels))
    width = 0.25
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(x - width, val_acc, width, label="best val acc")
    ax.bar(x, test_acc, width, label="test acc")
    ax.bar(x + width, f1, width, label="F1")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=20, ha="right")
    ax.set_ylim(0, 1)
    ax.set_ylabel("score")
    ax.set_title("Fine-tuning score comparison")
    ax.legend()
    ax.grid(axis="y", alpha=0.25)
    if save:
        save_current_figure("finetune_scores.png")
    plt.show()


def plot_finetune_accuracy(runs: list[dict], save: bool = True) -> None:
    """Backward-compatible alias for score comparison."""
    plot_finetune_scores(runs, save=save)


def plot_finetune_prf(runs: list[dict], save: bool = True) -> None:
    if not runs:
        print("No finetune runs to plot.")
        return
    labels = [r["summary"]["name"] for r in runs]
    precision = [r["summary"]["precision"] for r in runs]
    recall = [r["summary"]["recall"] for r in runs]
    f1 = [r["summary"]["f1"] for r in runs]
    x = np.arange(len(labels))
    width = 0.25
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(x - width, precision, width, label="precision")
    ax.bar(x, recall, width, label="recall")
    ax.bar(x + width, f1, width, label="F1")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=20, ha="right")
    ax.set_ylim(0, 1)
    ax.set_ylabel("score")
    ax.set_title("Precision/recall/F1 comparison")
    ax.legend()
    ax.grid(axis="y", alpha=0.25)
    if save:
        save_current_figure("finetune_prf.png")
    plt.show()


def plot_finetune_quality_vs_trainable_params(runs: list[dict], save: bool = True) -> None:
    if not runs:
        print("No finetune runs to plot.")
        return
    fig, ax = plt.subplots(figsize=(7, 5))
    for run in runs:
        s = run["summary"]
        ax.scatter(s["trainable_params"], s["test_accuracy"], s=80)
        ax.text(s["trainable_params"], s["test_accuracy"], " " + s["name"], va="center")
    ax.set_xlabel("trainable parameters")
    ax.set_ylabel("test accuracy")
    ax.set_title("Task quality vs trainable parameters")
    ax.grid(alpha=0.25)
    if save:
        save_current_figure("finetune_quality_vs_trainable_params.png")
    plt.show()


def plot_finetune_quality_vs_runtime(runs: list[dict], save: bool = True) -> None:
    if not runs:
        print("No finetune runs to plot.")
        return
    fig, ax = plt.subplots(figsize=(7, 5))
    for run in runs:
        s = run["summary"]
        ax.scatter(s["runtime_sec"], s["test_accuracy"], s=80)
        ax.text(s["runtime_sec"], s["test_accuracy"], " " + s["name"], va="center")
    ax.set_xlabel("runtime seconds")
    ax.set_ylabel("test accuracy")
    ax.set_title("Task quality vs runtime")
    ax.grid(alpha=0.25)
    if save:
        save_current_figure("finetune_quality_vs_runtime.png")
    plt.show()


def display_high_confidence_errors(runs: list[dict], limit: int = 10) -> None:
    rows = []
    for row in finetune_high_confidence_error_rows(runs)[:limit]:
        rows.append({
            "run": row["run"],
            "label": row["label"],
            "pred": row["pred"],
            "confidence": format_float(row["confidence"]),
            "text": row["text"],
        })
    display_table(rows, [("run", "run"), ("label", "label"), ("pred", "pred"), ("confidence", "confidence"), ("text", "text")], title="High-confidence errors")


## 16. 파인튜닝 비교 실행

**목적**
- `FINETUNE_EXPERIMENTS`에 정의된 감성분류 실험을 실행합니다.

**수정할 곳**
- 실행하려면 `ft_runs = run_finetune_many(FINETUNE_EXPERIMENTS)`의 주석을 해제합니다.
- 세 가지 freeze mode를 모두 비교하려면 `FINETUNE_SCOPE_EXPERIMENTS`를 실행합니다.

**출력**
- epoch별 train/val accuracy 로그
- finetune CSV 파일

**해석**
- best LM run이 자동으로 backbone으로 선택됩니다.
- full fine-tuning이 더 느린 만큼 성능 개선이 있는지 확인합니다.


In [ ]:
# 실행하려면 아래 줄의 주석을 해제하세요.
# ft_runs = run_finetune_many(FINETUNE_EXPERIMENTS)
# ft_scope_runs = run_finetune_many(FINETUNE_SCOPE_EXPERIMENTS)

best_lm = select_best_lm_run(RUNS)
print("준비된 파인튜닝 실험 수:", len(FINETUNE_EXPERIMENTS))
print("선택될 backbone:", best_lm["summary"]["name"] if best_lm else "scratch 또는 LM run 없음")
print("실행 명령: ft_runs = run_finetune_many(FINETUNE_EXPERIMENTS)")


## 17. Fine-tuning Results

**목적**
- fine-tuning 결과를 task score, 학습 곡선, confusion matrix, 오답 예시로 확인합니다.

**수정할 곳**
- 보통 수정하지 않습니다.
- 특정 run만 보고 싶으면 아래 코드 셀의 `for run in FINETUNE_RUNS:` 부분을 원하는 run으로 제한합니다.

**출력**
- score/loss/accuracy/precision-recall-F1 그래프.
- raw confusion matrix와 normalized confusion matrix.
- confidence가 높은 오답 표와 오류 방향 요약 파일.

**해석**
- accuracy와 F1을 함께 봅니다. 클래스별 오류가 한쪽으로 몰리면 F1이나 FP/FN breakdown이 accuracy보다 더 많은 정보를 줍니다.
- normalized confusion matrix는 클래스별 비율을 보여주므로 label 수가 불균형할 때 raw count보다 해석하기 쉽습니다.
- high-confidence errors는 모델이 강하게 확신했지만 틀린 사례입니다. 데이터 노이즈, 문장 길이, 부정 표현, 풍자 표현 같은 실패 패턴을 찾기에 좋습니다.


In [ ]:
display_result_data_guide("Fine-tuning")
display_finetune_summary(FINETUNE_RUNS)
plot_finetune_loss(FINETUNE_RUNS)
plot_finetune_accuracy_curve(FINETUNE_RUNS)
plot_finetune_scores(FINETUNE_RUNS)
plot_finetune_prf(FINETUNE_RUNS)
plot_finetune_quality_vs_trainable_params(FINETUNE_RUNS)
plot_finetune_quality_vs_runtime(FINETUNE_RUNS)
for run in FINETUNE_RUNS:
    matrix = np.array(run["confusion_matrix"])
    name = run["summary"]["name"]
    plot_confusion_matrix(matrix, name, save_name=f"confusion_{name}.png")
    plot_confusion_matrix(matrix, name, save_name=f"confusion_{name}_normalized.png", normalize=True)
display_high_confidence_errors(FINETUNE_RUNS)
export_finetune_artifacts(FINETUNE_RUNS)


## 18. Report Summary

**목적**
- 실행 결과를 보고서에 바로 옮길 수 있는 Markdown으로 정리합니다.

**수정할 곳**
- 보통 수정하지 않습니다.
- 보고서 문장 톤이나 표 컬럼을 바꾸고 싶으면 아래 코드 셀의 `build_report_markdown`을 수정합니다.

**출력**
- `result_data_guide.md`: 모든 결과 데이터의 의미와 해석 가이드.
- `lm_report_summary.md`: LM pretraining 전용 요약.
- `finetune_report_summary.md`: fine-tuning 전용 요약.
- `lab_report_summary.md`: 데이터 분포, LM, 생성문, fine-tuning, 오류 분석을 묶은 전체 실험 요약.

**해석**
- report는 결론만 고정하지 않습니다. 어떤 run이 좋은지, 왜 그런지, 다음 실험을 어디로 이어갈지까지 함께 남깁니다.
- classification metric은 `data_label_distribution.csv/png`로 label 분포를 확인한 뒤 해석합니다.


In [ ]:
def next_experiment_suggestions(lm_runs: list[dict], ft_runs: list[dict]) -> list[str]:
    """Create short follow-up ideas from the currently available results."""
    suggestions = []
    if lm_runs:
        best = min(lm_runs, key=lambda r: r["summary"]["best_val_loss"])
        suggestions.append(f"Retest `{best['summary']['name']}` with seeds 42/43/44 to estimate variance.")
        suggestions.append(f"Keep `{best['summary']['name']}` size fixed and vary only position encoding or activation.")
        suggestions.append("Compare a slightly longer run with the same config to separate undertraining from architecture effects.")
    else:
        suggestions.append("Run the SMOKE preset with `learned_gelu` and `sinusoidal_position` first.")
    if ft_runs:
        best_ft = max(ft_runs, key=lambda r: r["summary"].get("f1", 0.0))
        suggestions.append(f"For fine-tuning, narrow the learning-rate search around `{best_ft['summary']['name']}`.")
        suggestions.append("Inspect high-confidence errors before increasing model size; label noise can mimic model weakness.")
        suggestions.append("Check `data_label_distribution.csv` before interpreting accuracy-only gains.")
    else:
        suggestions.append("Fine-tune the best LM run with `classifier_only` and `full_finetune` as the first scope comparison.")
    return suggestions


def lm_learning_dynamics_notes(runs: list[dict]) -> list[str]:
    """Summarize loss/perplexity/gap patterns in report-friendly language."""
    if not runs:
        return ["No LM run has been executed yet."]
    notes = []
    for run in runs:
        summary = run["summary"]
        name = summary["name"]
        gap = summary.get("generalization_gap")
        best_loss = summary.get("best_val_loss")
        final_loss = summary.get("final_val_loss", best_loss)
        if gap is not None:
            if gap > 0.7:
                notes.append(f"`{name}` has a large validation-train gap ({gap:.3f}); check overfitting or a train/validation distribution mismatch.")
            elif gap < 0.1:
                notes.append(f"`{name}` has a small validation-train gap ({gap:.3f}); if both losses are high, it may need more capacity or steps.")
            else:
                notes.append(f"`{name}` has a moderate validation-train gap ({gap:.3f}); compare quality and runtime before changing several variables at once.")
        if final_loss is not None and best_loss is not None and final_loss > best_loss + 0.2:
            notes.append(f"`{name}` ended worse than its best validation point; consider shorter training or stronger regularization.")
    return notes[:8]


def compute_efficiency_rows(runs: list[dict]) -> list[dict]:
    """Format compute-oriented LM metrics for Markdown reports."""
    rows = []
    for row in lm_summary_rows(runs):
        rows.append({
            "run": row["run"],
            "params": f"{row['params']:,}",
            "runtime": row["runtime"],
            "tokens_per_sec": "" if row.get("tokens_per_sec") is None else f"{row['tokens_per_sec']:.0f}",
            "best_val_loss": format_float(row["best_val_loss"], 3),
            "best_perplexity": format_float(row["best_perplexity"], 1),
        })
    return rows


def generation_review_rows_for_report(runs: list[dict], limit: int = 12) -> list[dict]:
    """Collect a compact sample of generation rows for the combined report."""
    rows = []
    for row in generation_rows(runs):
        rows.append({
            "run": row.get("run"),
            "phase": row.get("phase"),
            "decoding": row.get("decoding"),
            "prompt": row.get("prompt"),
            "generated_text": row.get("generated_text"),
        })
        if len(rows) >= limit:
            break
    return rows


def data_distribution_report_rows() -> list[dict]:
    """Format label distribution rows for the combined Markdown report."""
    if "data_label_distribution_rows" not in globals():
        return []
    rows = data_label_distribution_rows()
    by_split = {}
    for row in rows:
        split = row["split"]
        by_split.setdefault(split, {"split": split, "negative": 0, "positive": 0, "positive_share": row.get("positive_share"), "majority_share": row.get("majority_share"), "imbalance_ratio": row.get("imbalance_ratio")})
        if row["label"] == 0:
            by_split[split]["negative"] = row["count"]
        elif row["label"] == 1:
            by_split[split]["positive"] = row["count"]
    formatted = []
    for split, row in by_split.items():
        formatted.append({
            "split": split,
            "negative": row["negative"],
            "positive": row["positive"],
            "positive_share": format_float(row.get("positive_share")),
            "majority_share": format_float(row.get("majority_share")),
            "imbalance_ratio": format_float(row.get("imbalance_ratio")),
        })
    return formatted


def error_analysis_rows_for_report(ft_runs: list[dict]) -> list[dict]:
    """Combine FP/FN counts with high-confidence error counts for reports."""
    high_conf_rows = finetune_high_confidence_error_rows(ft_runs, threshold=0.8)
    high_conf_counts = {}
    for row in high_conf_rows:
        high_conf_counts[row["run"]] = high_conf_counts.get(row["run"], 0) + 1
    rows = []
    for run in ft_runs:
        s = run["summary"]
        rows.append({
            "run": s["name"],
            "fp": s.get("fp"),
            "fn": s.get("fn"),
            "high_conf_errors": high_conf_counts.get(s["name"], 0),
            "note": "more FP" if s.get("fp", 0) > s.get("fn", 0) else "more FN" if s.get("fn", 0) > s.get("fp", 0) else "balanced",
        })
    return rows


def build_report_markdown(lm_runs: list[dict], ft_runs: list[dict]) -> str:
    """Build the full report summary with a stable, report-friendly section order."""
    lines = []
    lines.append("## Run Overview")
    lines.append("")
    lines.append(f"- tokenizer preset: `{globals().get('TOKENIZER_PRESET', '')}`")
    lines.append(f"- tokenized data preset: `{globals().get('TOKENIZED_DATA_PRESET', '')}`")
    lines.append(f"- training preset: `{globals().get('TRAINING_PRESET', RUN_PRESET)}`")
    lines.append(f"- fine-tuning preset: `{globals().get('FINETUNE_PRESET', '')}`")
    lines.append(f"- artifact dir: `{ARTIFACT_DIR}`")
    lines.append(f"- LM runs: `{len(lm_runs)}`")
    lines.append(f"- fine-tuning runs: `{len(ft_runs)}`")
    lines.append("")

    lines.append("## Data Overview")
    lines.append("")
    data_rows = data_distribution_report_rows()
    if data_rows:
        lines.extend(markdown_table(data_rows, [("split", "split"), ("negative", "negative"), ("positive", "positive"), ("positive_share", "positive share"), ("majority_share", "majority share"), ("imbalance_ratio", "imbalance ratio")]))
        lines.append("")
        lines.append("Interpret classification accuracy together with precision, recall, F1, and confusion matrices when label balance differs by split.")
    else:
        lines.append("No data label distribution is available yet.")
    lines.append("")

    lines.append("## LM Pretraining Summary")
    lines.append("")
    if lm_runs:
        lines.append(f"Reference run: `{reference_label_for_runs(lm_runs)}`")
        lines.append("")
        lines.extend(markdown_table(lm_summary_rows(lm_runs), [
            ("run", "run"),
            ("config_summary", "config"),
            ("best_val_loss", "best val loss"),
            ("reference_delta", "reference delta"),
            ("best_perplexity", "best ppl"),
            ("params", "params"),
            ("runtime", "runtime"),
            ("tokens_per_sec", "tokens/sec"),
        ]))
        lines.append("")
        lines.extend(markdown_table(lm_reference_comparison_rows(lm_runs), [
            ("run", "run"),
            ("reference_run", "reference"),
            ("best_val_loss", "best val loss"),
            ("reference_best_val_loss", "reference loss"),
            ("reference_delta", "delta"),
            ("relative_delta_pct", "delta %"),
        ]))
        lines.append("")
        best_lm = min(lm_runs, key=lambda r: r["summary"]["best_val_loss"])
        s = best_lm["summary"]
        lines.append(f"Best LM run: `{s['name']}` with validation loss `{s['best_val_loss']:.3f}` and perplexity `{perplexity(s['best_val_loss']):.1f}`.")
        lines.append("")
        lines.append("Compute efficiency:")
        lines.extend(markdown_table(compute_efficiency_rows(lm_runs), [
            ("run", "run"),
            ("params", "params"),
            ("runtime", "runtime"),
            ("tokens_per_sec", "tokens/sec"),
            ("best_val_loss", "best val loss"),
            ("best_perplexity", "best ppl"),
        ]))
    else:
        lines.append("No LM run has been executed yet.")
    lines.append("")

    lines.append("## LM Learning Dynamics")
    lines.append("")
    for note in lm_learning_dynamics_notes(lm_runs):
        lines.append(f"- {note}")
    lines.append("")

    lines.append("## Generation Review")
    lines.append("")
    generation_review = generation_review_rows_for_report(lm_runs)
    if generation_review:
        lines.append("Generation samples are qualitative evidence. Keep prompt and decoding fixed, then compare them with validation loss and perplexity.")
        lines.append("")
        lines.extend(markdown_table(generation_review, [
            ("run", "run"),
            ("phase", "phase"),
            ("decoding", "decoding"),
            ("prompt", "prompt"),
            ("generated_text", "generated text"),
        ]))
    else:
        lines.append("No generation sample is available yet.")
    lines.append("")

    lines.append("## Fine-tuning Summary")
    lines.append("")
    if ft_runs:
        lines.extend(markdown_table(finetune_summary_rows(ft_runs), [
            ("run", "run"),
            ("source_lm_run", "LM source"),
            ("freeze_mode", "freeze mode"),
            ("selected_epoch", "selected epoch"),
            ("best_val_loss", "best val loss"),
            ("best_val_accuracy", "best val acc"),
            ("test_accuracy", "test acc"),
            ("precision", "precision"),
            ("recall", "recall"),
            ("f1", "F1"),
            ("fp", "FP"),
            ("fn", "FN"),
            ("runtime", "runtime"),
        ]))
    else:
        lines.append("No fine-tuning run has been executed yet.")
    lines.append("")

    lines.append("## Error Analysis")
    lines.append("")
    if ft_runs:
        lines.append("Use raw confusion matrices for counts and normalized matrices for per-class error rates.")
        lines.append("")
        lines.extend(markdown_table(error_analysis_rows_for_report(ft_runs), [
            ("run", "run"),
            ("fp", "FP"),
            ("fn", "FN"),
            ("high_conf_errors", "high-confidence errors"),
            ("note", "direction"),
        ]))
        lines.append("")
        lines.extend(markdown_table(finetune_error_breakdown_rows(ft_runs), [
            ("run", "run"),
            ("error_type", "error type"),
            ("count", "count"),
            ("share", "share"),
        ]))
    else:
        lines.append("Run fine-tuning first to inspect FP/FN and high-confidence errors.")
    lines.append("")

    lines.append("## Next Experiments")
    lines.append("")
    for item in next_experiment_suggestions(lm_runs, ft_runs):
        lines.append(f"- {item}")
    lines.append("")
    return "\n".join(lines)


display_result_data_guide("Report/Guide")
report_md = build_report_markdown(RUNS, FINETUNE_RUNS)
display(Markdown(report_md))
write_markdown(ARTIFACT_DIR / "lab_report_summary.md", report_md)
if RUNS:
    write_markdown(ARTIFACT_DIR / "lm_report_summary.md", build_lm_report_markdown(RUNS))
if FINETUNE_RUNS:
    write_markdown(ARTIFACT_DIR / "finetune_report_summary.md", build_finetune_report_markdown(FINETUNE_RUNS))
export_result_data_guide()
